***
#### Exercício Comparativo — Hipótese H1-Rendimentos (`modelos_fixos_h1_rendimentos_v1.ipynb`)

**Hipótese testada neste caderno (H1-Rendimentos):**
> *"A associação entre os rendimentos reais de soja e milho e a taxa de conversão da vegetação nativa para agricultura é mais positiva nos municípios com maior capacidade produtiva potencial da respectiva cultura, mantidas as demais condições relevantes."*

##### 2.2 Diagnóstico do Modelo Inicial, Definições Operacionais de Rendimentos Reais e Linha de Base (Etapas 1 a 3)

**Decisões metodológicas documentadas (`/grill-me`):**
1. **Variável principal de rendimentos reais por hectare (`rec_s_l1`, `rec_m_l1`):** receita bruta real estimada por hectare colhido em $t-1$ (em USD de 2024/ha, colunas `x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag1__soja` e `...__milho`), correspondente ao produto entre o preço real anual ($P_{c,t-1}$) e a produtividade física municipal efetivamente realizada na PAM ($Y^{\text{PAM}}_{i,c,t-1}$, em t/ha).
2. **Dupla variação ($i, t$), inclusão do efeito principal e tratamento dos zeros de cultivo em $t-1$:**
   - Diferentemente do preço anual $P_{c,t-1}$ (que varia apenas em $t$ e cujo efeito principal é absorvido pelos efeitos fixos de ano ou `bioma^ano`), a receita real por hectare varia entre municípios e anos ($i, t$). Por isso, a especificação correta inclui **tanto o efeito principal do rendimento real por hectare quanto sua interação com a capacidade produtiva potencial padronizada ($Z_{i,c}$)**, reportando-se também a versão apenas com a interação para comparação direta com `modelos_fixos_h1_prec_v2.ipynb`.
   - Na amostra observada de conversão (`obs = TRUE`, $N = 2.772$), há 665 observações com zero colheita de soja em $t-1$ ($2.107$ positivas) e 67 com zero colheita de milho em $t-1$ ($2.705$ positivas). Estimamos os modelos em duas amostras complementares:
     - **(a) Subamostra com lavoura ativa em $t-1$ (`rec_l1 > 0`):** usando o logaritmo natural exato centrado (`lrs_act = ln(rec_s_l1) - media`, `xrs_act = lrs_act * zs`), onde vale a identidade aditiva exata $\ln(\text{receita/ha}_{i,t-1}) \equiv \ln(P_{t-1}) + \ln(Y^{\text{PAM, t/ha}}_{i,t-1})$;
     - **(b) Amostra observada completa (`obs = TRUE`, $N = 2.772$):** usando $\ln(1 + \text{receita/ha}_{i,t-1})$ centrado (`lrs_1p`, `xrs_1p`), com e sem controle explícito para a margem extensiva de cultivo zero em $t-1$ (`zero_s_l1`), além de especificações em nível (USD 1.000/ha) e em seno hiperbólico inverso ($\text{asinh}$).
3. **Variáveis comparativas (decomposição Preço vs. Produtividade Física PAM e Lucro Operacional Líquido Real/ha):**
   - Decomposição direta de $\widetilde{\ln R_{i,t-1}} \times Z_i$ nos componentes de preço externo ($\widetilde{\ln P_{t-1}} \times Z_i$) e produtividade física realizada ($\widetilde{\ln Y^{\text{PAM}}_{i,t-1}} \times Z_i$);
   - Lucro operacional líquido real por hectare em $t-1$: $\text{receita bruta real/ha}_{i,t-1} - \text{custo operacional CONAB real/ha}_{t-1}$ (em USD 1.000/ha e em $\text{asinh}$).


In [1]:
# Pacotes para estimação em painel (fixest), inferência HAC, leitura rápida e geoprocessamento
suppressPackageStartupMessages({
  library(tidyverse)
  library(fixest)
  library(sandwich)
  library(lmtest)
  library(data.table)
  library(sf)
})
options(width = 150)
setFixest_nthreads(1)
setFixest_notes(FALSE)
sf_use_s2(TRUE)

# Diretório de trabalho e pasta de auditoria exclusiva para o exercício de rendimentos
dir_modelos <- "C:/Users/ferna/Documents/dev/Tese USP/tese_eco_usp/2_INTI/chapters/chap1/stats/2. MODELOS/grupos_h1/2_base_completa_h1_precaria/MODELOS_H1_PREC"
if (!file.exists("base_modelos_h1_prec.csv")) setwd(dir_modelos)
root <- normalizePath(file.path(getwd(), "../../../../../../.."), winslash = "/")
stopifnot(basename(root) == "2_INTI")
out_dir <- file.path(root, "audit", "modelos_h1_rendimentos_20261110")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

tf <- function(x) tolower(as.character(x)) == "true"
sd_pop <- function(x) sqrt(mean((x - mean(x, na.rm = TRUE))^2, na.rm = TRUE))

raw_r1 <- as_tibble(read.csv("base_modelos_h1_prec.csv", stringsAsFactors = FALSE))
stopifnot(nrow(raw_r1) == 3384, !anyDuplicated(raw_r1[c("z0_codigo_ibge", "z0_ano")]))

df_r1 <- raw_r1 %>% transmute(
  ibge = factor(z0_codigo_ibge),
  ibge_int = as.integer(z0_codigo_ibge),
  ano = as.integer(z0_ano),
  micro = factor(z0_microrregiao),
  bioma = factor(z0_bioma_predominante),
  amz = z0_biome_amazonia_share_i,
  cer = z0_biome_cerrado_share_i,
  pan = z0_biome_pantanal_share_i,
  taxa_agr = y1a_taxa_conversao__agricultura,
  y_agr = 100 * y1a_taxa_conversao__agricultura,
  y_pas = 100 * y1a_taxa_conversao__pastagem,
  area_soja_ha = x2b_area_plantada_ha__soja,
  area_milho_ha = x2b_area_plantada_ha__milho,
  colh_soja_ha = x2b_area_colhida_ha__soja,
  colh_milho_ha = x2b_area_colhida_ha__milho,
  ps_l1 = x2a_preco_real_usd2024_media_anual_lag1__soja,
  pm_l1 = x2a_preco_real_usd2024_media_anual_lag1__milho,
  ps_t  = x2a_preco_real_usd2024_media_anual__soja,
  pm_t  = x2a_preco_real_usd2024_media_anual__milho,
  dap_l1 = x2a_preco_real_usd2024_media_anual_lag1__dap,
  rec_s_l1 = x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag1__soja,
  rec_m_l1 = x2c_receita_bruta_estimada_usd2024_por_ha_colhido_lag1__milho,
  rec_s_t  = x2c_receita_bruta_estimada_usd2024_por_ha_colhido__soja,
  rec_m_t  = x2c_receita_bruta_estimada_usd2024_por_ha_colhido__milho,
  apt_s = x2b_capacidade_valor__soja__gaez_yxHr0_soy,
  apt_m = x2b_capacidade_valor__milho__gaez_yxHr0_mze,
  cst_s = x2d_custo_conab_soja_ogm_safra1_conab_h_usd2024,
  cst_m = x2d_custo_conab_milho_2a_ogm_safra0_conab_h_usd2024,
  rain = x1c_v2__precip_anomaly_1981_2000 / 100,
  autos_l1 = x3_n_autos_flora_elegiveis_lag1__ibama_espacial,
  obs = tf(x4_v2__flag_amostra_conversao_ambos),
  fisc = tf(x4_v2__flag_amostra_fiscal_lag1)
)

# Séries anuais (24 anos) e cadastro municipal (141 municípios)
anos_r1 <- df_r1 %>% distinct(ano, ps_l1, pm_l1, ps_t, pm_t, dap_l1) %>% arrange(ano) %>%
  mutate(ps_lead = lead(ps_t), pm_lead = lead(pm_t))
mun_r1 <- df_r1 %>% distinct(ibge, ibge_int, apt_s, apt_m)
stopifnot(nrow(anos_r1) == 24, nrow(mun_r1) == 141)

c_ps <- mean(log(anos_r1$ps_l1)); c_pm <- mean(log(anos_r1$pm_l1)); c_dap <- mean(log(anos_r1$dap_l1))
# Médias de referência na amostra observada para centragem das variáveis de receita e produtividade
c_lrs_act <- mean(log(df_r1$rec_s_l1[df_r1$obs & df_r1$rec_s_l1 > 0]))
c_lrm_act <- mean(log(df_r1$rec_m_l1[df_r1$obs & df_r1$rec_m_l1 > 0]))
c_lrs_1p  <- mean(log1p(df_r1$rec_s_l1[df_r1$obs]))
c_lrm_1p  <- mean(log1p(df_r1$rec_m_l1[df_r1$obs]))
c_rs_lvl  <- mean(df_r1$rec_s_l1[df_r1$obs])
c_rm_lvl  <- mean(df_r1$rec_m_l1[df_r1$obs])
c_asinh_s <- mean(asinh(df_r1$rec_s_l1[df_r1$obs]))
c_asinh_m <- mean(asinh(df_r1$rec_m_l1[df_r1$obs]))

# Produtividade física implícita em t-1 (em t/ha): rec_l1 / ps_l1 (exata para todos os 24 anos, inclusive 2001)
df_r1 <- df_r1 %>%
  left_join(anos_r1 %>% select(ano, ps_lead, pm_lead), by = "ano") %>%
  arrange(ibge, ano) %>%
  group_by(ibge) %>%
  mutate(
    cst_s_l1   = lag(cst_s),
    cst_m_l1   = lag(cst_m),
    rec_s_lead = lead(rec_s_t),
    rec_m_lead = lead(rec_m_t)
  ) %>%
  ungroup() %>%
  mutate(
    # Aptidão padronizada (Z) e preços centrados (para comparação e decomposição)
    zs = (apt_s - mean(mun_r1$apt_s)) / sd_pop(mun_r1$apt_s),
    zm = (apt_m - mean(mun_r1$apt_m)) / sd_pop(mun_r1$apt_m),
    lps = log(ps_l1) - c_ps,
    lpm = log(pm_l1) - c_pm,
    xs = lps * zs,
    xm = lpm * zm,
    trend = (ano - 2001) / 10,
    zs_tr = zs * trend,
    zm_tr = zm * trend,

    # 1. Subamostra com lavoura ativa em t-1 (rec_l1 > 0): log natural exato
    act_s = rec_s_l1 > 0,
    act_m = rec_m_l1 > 0,
    lrs_act = ifelse(act_s, log(rec_s_l1) - c_lrs_act, NA_real_),
    lrm_act = ifelse(act_m, log(rec_m_l1) - c_lrm_act, NA_real_),
    xrs_act = lrs_act * zs,
    xrm_act = lrm_act * zm,

    # Decomposição aditiva exata na amostra ativa: ln(rec_l1) = ln(P_l1) + ln(Y_PAM_t_ha_l1)
    yld_s_tha_l1 = ifelse(act_s, rec_s_l1 / ps_l1, 0),
    yld_m_tha_l1 = ifelse(act_m, rec_m_l1 / pm_l1, 0),
    lyld_s_act = ifelse(act_s, log(yld_s_tha_l1) - mean(log(yld_s_tha_l1[obs & act_s]), na.rm = TRUE), NA_real_),
    lyld_m_act = ifelse(act_m, log(yld_m_tha_l1) - mean(log(yld_m_tha_l1[obs & act_m]), na.rm = TRUE), NA_real_),
    xyld_s_act = lyld_s_act * zs,
    xyld_m_act = lyld_m_act * zm,

    # 2. Amostra completa (incluindo zeros): ln(1 + rec_l1), nível (USD 1.000/ha) e asinh
    zero_s_l1 = as.numeric(rec_s_l1 == 0),
    zero_m_l1 = as.numeric(rec_m_l1 == 0),
    zero_s_zs = zero_s_l1 * zs,
    zero_m_zm = zero_m_l1 * zm,
    lrs_1p = log1p(rec_s_l1) - c_lrs_1p,
    lrm_1p = log1p(rec_m_l1) - c_lrm_1p,
    xrs_1p = lrs_1p * zs,
    xrm_1p = lrm_1p * zm,
    rs_k   = (rec_s_l1 - c_rs_lvl) / 1000,
    rm_k   = (rec_m_l1 - c_rm_lvl) / 1000,
    xrs_k  = rs_k * zs,
    xrm_k  = rm_k * zm,
    lrs_asinh = asinh(rec_s_l1) - c_asinh_s,
    lrm_asinh = asinh(rec_m_l1) - c_asinh_m,
    xrs_asinh = lrs_asinh * zs,
    xrm_asinh = lrm_asinh * zm,

    # 3. Lucro operacional líquido real por ha em t-1 (USD 1.000/ha e asinh)
    luc_s_ha_raw = ifelse(act_s, rec_s_l1 - cst_s_l1, 0),
    luc_m_ha_raw = ifelse(act_m, rec_m_l1 - cst_m_l1, 0),
    luc_s_act_k  = ifelse(act_s, (rec_s_l1 - cst_s_l1 - mean(rec_s_l1[obs & act_s] - cst_s_l1[obs & act_s], na.rm = TRUE)) / 1000, NA_real_),
    luc_m_act_k  = ifelse(act_m, (rec_m_l1 - cst_m_l1 - mean(rec_m_l1[obs & act_m] - cst_m_l1[obs & act_m], na.rm = TRUE)) / 1000, NA_real_),
    xluc_s_act_k = luc_s_act_k * zs,
    xluc_m_act_k = luc_m_act_k * zm,
    luc_s_full_k = (luc_s_ha_raw - mean(luc_s_ha_raw[obs], na.rm = TRUE)) / 1000,
    luc_m_full_k = (luc_m_ha_raw - mean(luc_m_ha_raw[obs], na.rm = TRUE)) / 1000,
    xluc_s_full_k = luc_s_full_k * zs,
    xluc_m_full_k = luc_m_full_k * zm,
    luc_s_asinh  = asinh(luc_s_ha_raw / 100) - mean(asinh(luc_s_ha_raw[obs] / 100), na.rm = TRUE),
    luc_m_asinh  = asinh(luc_m_ha_raw / 100) - mean(asinh(luc_m_ha_raw[obs] / 100), na.rm = TRUE),
    xluc_s_asinh = luc_s_asinh * zs,
    xluc_m_asinh = luc_m_asinh * zm,

    # Placebo temporal (receita/ha em t+1)
    lrs_act_f = ifelse(rec_s_lead > 0, log(rec_s_lead) - c_lrs_act, NA_real_),
    lrm_act_f = ifelse(rec_m_lead > 0, log(rec_m_lead) - c_lrm_act, NA_real_),
    xrs_act_f = lrs_act_f * zs,
    xrm_act_f = lrm_act_f * zm,
    lrs_1p_f  = log1p(rec_s_lead) - c_lrs_1p,
    lrm_1p_f  = log1p(rec_m_lead) - c_lrm_1p,
    xrs_1p_f  = lrs_1p_f * zs,
    xrm_1p_f  = lrm_1p_f * zm,

    # Moderação por bioma (shares contínuos, referência = Cerrado)
    xrs_act_amz = xrs_act * amz, xrs_act_pan = xrs_act * pan,
    xrm_act_amz = xrm_act * amz, xrm_act_pan = xrm_act * pan,
    xrs_1p_amz  = xrs_1p * amz,  xrs_1p_pan  = xrs_1p * pan,
    xrm_1p_amz  = xrm_1p * amz,  xrm_1p_pan  = xrm_1p * pan,

    # Custos e fiscalização na amostra fiscal
    x_dap_s = (log(dap_l1) - c_dap) * zs,
    x_dap_m = (log(dap_l1) - c_dap) * zm,
    x_cst_s = (log(cst_s_l1) - mean(log(cst_s_l1), na.rm = TRUE)) * zs,
    x_cst_m = (log(cst_m_l1) - mean(log(cst_m_l1), na.rm = TRUE)) * zm,
    f1 = log1p(autos_l1),
    f1_rs_act = lrs_act * f1, f1_zs = zs * f1, f1_xrs_act = xrs_act * f1,
    f1_rm_act = lrm_act * f1, f1_zm = zm * f1, f1_xrm_act = xrm_act * f1,
    f1_rs_1p  = lrs_1p * f1,  f1_xrs_1p  = xrs_1p * f1,
    f1_rm_1p  = lrm_1p * f1,  f1_xrm_1p  = xrm_1p * f1,
    post05 = as.numeric(ano >= 2005),
    post06 = as.numeric(ano >= 2006)
  )

d_obs   <- df_r1 %>% filter(obs)
d_act_s <- d_obs %>% filter(act_s)
d_act_m <- d_obs %>% filter(act_m)
d_fisc  <- df_r1 %>% filter(fisc)
stopifnot(nrow(d_obs) == 2772, nrow(d_act_s) == 2107, nrow(d_act_m) == 2705, nrow(d_fisc) == 1140)

bioma_amz <- levels(df_r1$bioma)[grepl("^Amaz", levels(df_r1$bioma))]
bioma_cer <- levels(df_r1$bioma)[grepl("^Cerr", levels(df_r1$bioma))]
bioma_pan <- levels(df_r1$bioma)[grepl("^Pant", levels(df_r1$bioma))]

RES <- list()
rec <- function(m, id, grupo, descr, amostra = NA_character_) {
  ct <- as.data.frame(coeftable(m))
  RES[[length(RES) + 1]] <<- tibble(
    id = id, grupo = grupo, descricao = descr, amostra = amostra,
    termo = rownames(ct), estimativa = ct[[1]], erro_padrao = ct[[2]], p = ct[[4]], n = nobs(m)
  )
  invisible(m)
}

# Função padrão de estimação para H1-Rendimentos:
# spec = 'act' usa lrs_act + xrs_act na amostra com lavoura ativa (rec_l1 > 0)
# spec = '1p'  usa lrs_1p  + xrs_1p  na amostra observada completa (obs = TRUE)
fit_run <- function(crop, data = NULL, spec = "act", include_main = TRUE, y = "y_agr",
                    fe = "ibge + bioma^ano", vcov = ~ibge, trend = TRUE, rain = TRUE, extra = NULL) {
  if (is.null(data)) data <- if (spec == "act") (if (crop == "soja") d_act_s else d_act_m) else d_obs
  if (spec == "act") {
    lv <- if (crop == "soja") "lrs_act" else "lrm_act"
    x  <- if (crop == "soja") "xrs_act" else "xrm_act"
  } else {
    lv <- if (crop == "soja") "lrs_1p" else "lrm_1p"
    x  <- if (crop == "soja") "xrs_1p" else "xrm_1p"
  }
  ztr <- if (crop == "soja") "zs_tr" else "zm_tr"
  rhs <- c(if (include_main) lv, x, if (trend) ztr, if (rain) "rain", extra)
  f <- as.formula(paste(y, "~", paste(rhs, collapse = " + "), "|", fe))
  feols(f, data = data, vcov = vcov)
}
run <- function(crop, id, grupo, descr, amostra, ...) {
  rec(fit_run(crop, ...), id, grupo, descr, amostra)
}
tab <- function(..., headers = NULL) {
  ms <- list(...)
  if (is.null(headers)) headers <- paste0("(", seq_along(ms), ")")
  estrelas <- function(p) ifelse(p < 0.01, "***", ifelse(p < 0.05, "**", ifelse(p < 0.10, "*", "")))
  termos <- unique(unlist(lapply(ms, function(m) rownames(coeftable(m)))))
  col <- function(m) {
    ct <- coeftable(m)
    vapply(termos, function(t) {
      if (t %in% rownames(ct)) {
        sprintf("%s%s (%s)", formatC(ct[t, 1], digits = 3, format = "g", flag = "#"),
                estrelas(ct[t, 4]), formatC(ct[t, 2], digits = 3, format = "g", flag = "#"))
      } else ""
    }, character(1))
  }
  out <- as.data.frame(do.call(cbind, lapply(ms, col)), stringsAsFactors = FALSE)
  colnames(out) <- headers; rownames(out) <- termos
  info <- do.call(cbind, lapply(ms, function(m) c(
    `N` = format(nobs(m), big.mark = ""),
    `R2 within` = sprintf("%.4f", as.numeric(r2(m, "wr2"))),
    `Efeitos fixos` = paste(m$fixef_vars, collapse = " + "))))
  colnames(info) <- headers
  print(rbind(out, as.data.frame(info, stringsAsFactors = FALSE)))
  cat("Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01\n")
}
cat(sprintf("Ambiente H1-Rendimentos configurado.\nAmostra observada: %d obs (%d mun.) | Lavoura ativa t-1: Soja = %d obs (%d mun.), Milho = %d obs (%d mun.).\n",
            nrow(d_obs), n_distinct(d_obs$ibge), nrow(d_act_s), n_distinct(d_act_s$ibge), nrow(d_act_m), n_distinct(d_act_m$ibge)))


Ambiente H1-Rendimentos configurado.
Amostra observada: 2772 obs (137 mun.) | Lavoura ativa t-1: Soja = 2107 obs (129 mun.), Milho = 2705 obs (137 mun.).


###### Transição passo a passo da linha de base para a especificação padronizada (Rendimentos Reais/ha)
As tabelas abaixo mostram o efeito isolado de cada etapa quando substituímos `preco_lag1` por `rec_l1` (receita bruta real/ha em $t-1$):
- **(1) original:** `rec_l1 * apt + autos_l1 + rec_l1 * autos_l1` na amostra preenchida ($N = 3.384$, níveis);
- **(2) Y observada:** restringe à amostra observada (`obs = TRUE`, $N = 2.769$);
- **(3) sem autos:** retira `autos_l1` e `risco`;
- **(4) ln(rec) s/ ef. princ. (ativa):** passa para $Y$ em p.p. e $\widetilde{\ln(\text{rec/ha}_{t-1})} \times Z_i$ na subamostra com lavoura ativa em $t-1$ (`rec_l1 > 0`), sem o efeito principal de `lrs_act` (`+ rain`);
- **(5) ln(rec) c/ ef. princ. (ativa):** inclui o efeito principal `lrs_act` (`lrm_act`), necessário porque a receita/ha varia por município e ano;
- **(6) ln(1+rec) c/ ef. princ. (obs):** estima na amostra observada completa ($N = 2.769$) usando $\ln(1 + \text{rec/ha}_{t-1})$.


In [2]:
nb_base_r <- df_r1 %>% mutate(
  taxa = taxa_agr,
  pot_rs = rec_s_l1 * apt_s, pot_rm = rec_m_l1 * apt_m,
  risco_rs = rec_s_l1 * autos_l1, risco_rm = rec_m_l1 * autos_l1
)

baseline_step_r <- function(crop) {
  p <- if (crop == "soja") {
    list(pot = "pot_rs", risco = "risco_rs", lv_a = "lrs_act", x_a = "xrs_act", lv_1p = "lrs_1p", x_1p = "xrs_1p", d_a = filter(nb_base_r, obs & act_s))
  } else {
    list(pot = "pot_rm", risco = "risco_rm", lv_a = "lrm_act", x_a = "xrm_act", lv_1p = "lrm_1p", x_1p = "xrm_1p", d_a = filter(nb_base_r, obs & act_m))
  }
  fm <- function(lhs, rhs) as.formula(paste(lhs, "~", rhs, "| ibge + ano"))
  m <- list(
    "1 original"         = feols(fm("taxa", paste(p$pot, "+ autos_l1 +", p$risco)), nb_base_r, vcov = ~ibge),
    "2 Y observada"      = feols(fm("taxa", paste(p$pot, "+ autos_l1 +", p$risco)), filter(nb_base_r, obs), vcov = ~ibge),
    "3 sem autos"        = feols(fm("taxa", p$pot), filter(nb_base_r, obs), vcov = ~ibge),
    "4 ativa s/ princ."  = feols(fm("y_agr", paste(p$x_a, "+ rain")), p$d_a, vcov = ~ibge),
    "5 ativa c/ princ."  = feols(fm("y_agr", paste(p$lv_a, "+", p$x_a, "+ rain")), p$d_a, vcov = ~ibge),
    "6 obs ln(1+rec)"    = feols(fm("y_agr", paste(p$lv_1p, "+", p$x_1p, "+ rain")), filter(nb_base_r, obs), vcov = ~ibge)
  )
  for (i in seq_along(m)) {
    am <- if (i == 1) "todas" else if (i %in% c(4, 5)) "ativa" else "obs"
    rec(m[[i]], paste0("B", i, "_", crop), "baseline", paste(crop, names(m)[i]), am)
  }
  cat("=== Linha de base (Rendimentos Reais/ha):", toupper(crop), "===\n")
  tab(m[[1]], m[[2]], m[[3]], m[[4]], m[[5]], m[[6]], headers = names(m))
}
baseline_step_r("soja")
baseline_step_r("milho")


=== Linha de base (Rendimentos Reais/ha): SOJA ===
                          1 original          2 Y observada           3 sem autos   4 ativa s/ princ.   5 ativa c/ princ.      6 obs ln(1+rec)
pot_rs         8.51e-11** (3.28e-11)  8.32e-11** (3.41e-11) 6.65e-11** (3.03e-11)                                                             
autos_l1      1.54e-05*** (4.86e-06) 1.74e-05*** (5.88e-06)                                                                                   
risco_rs      -8.77e-09** (3.83e-09) -1.13e-08** (5.16e-09)                                                                                   
xrs_act                                                                             0.152*** (0.0545)    0.148** (0.0571)                     
rain                                                                              -0.0172** (0.00742) -0.0171** (0.00745)  -0.0148** (0.00683)
lrs_act                                                                                    

=== Linha de base (Rendimentos Reais/ha): MILHO ===
                          1 original          2 Y observada          3 sem autos  4 ativa s/ princ.  5 ativa c/ princ.     6 obs ln(1+rec)
pot_rm          -3.37e-11 (5.21e-11)   -6.56e-11 (5.84e-11) -8.68e-11 (6.07e-11)                                                          
autos_l1      1.39e-05*** (4.69e-06) 1.93e-05*** (6.84e-06)                                                                               
risco_rm       -1.02e-08* (5.51e-09)  -1.68e-08* (8.80e-09)                                                                               
xrm_act                                                                          0.0615*** (0.0182) 0.0629*** (0.0182)                    
rain                                                                             -0.0130* (0.00675) -0.0129* (0.00671) -0.0147** (0.00693)
lrm_act                                                                                               -0.0503 (0.0

***
##### 2.3 Modelos Principais (M1 e M2), Decomposição Preço vs. Produtividade Física, Lucro Líquido/ha, Moderação por Bioma e Colinearidade (M3) (Etapas 4, 5 e 6)

**Estrutura da análise principal:**
1. **Painel A — Especificações principais de Receita Bruta Real/ha (`ibge + bioma^ano`, controle de tendência diferencial `zs_tr`/`zm_tr` e `rain`):**
   - `M1_act` / `M2_act`: subamostra com lavoura ativa em $t-1$ (`rec_l1 > 0`), incluindo efeito principal (`lrs_act`/`lrm_act`) e interação com aptidão (`xrs_act`/`xrm_act`);
   - `M1_act_nolv` / `M2_act_nolv`: subamostra com lavoura ativa em $t-1$, apenas com a interação (`xrs_act`/`xrm_act`), sem efeito principal;
   - `M1_1p` / `M2_1p`: amostra observada completa ($N = 2.767$), usando $\ln(1 + \text{rec/ha}_{t-1})$ (`lrs_1p + xrs_1p`);
   - `M1_1p_zero` / `M2_1p_zero`: amostra observada completa ($N = 2.767$), controlando adicionalmente o indicador de lavoura ausente em $t-1$ (`zero_l1` e `zero_l1 * Z`);
   - `M1_lvl` / `M2_lvl`: amostra observada completa ($N = 2.767$), em nível (USD 1.000/ha, `rs_k + xrs_k`);
   - `M1_asinh` / `M2_asinh`: amostra observada completa ($N = 2.767$), usando $\text{asinh}(\text{rec/ha}_{t-1})$.
2. **Painel B — Decomposição Exata (Preço Real $\ln P_{t-1}$ vs. Produtividade Física PAM $\ln Y^{\text{PAM}}_{i,t-1}$) e Lucro Operacional Líquido Real/ha (`receita/ha - custo CONAB/ha`):**
   - Como $\ln(\text{rec/ha}_{i,t-1}) = \ln(P_{t-1}) + \ln(Y^{\text{PAM, t/ha}}_{i,t-1})$ na amostra ativa, decompomos a interação em `xs` ($\ln P_{t-1} \times Z_i$) e `xyld_s_act` ($\ln Y^{\text{PAM}}_{i,t-1} \times Z_i$) para verificar qual componente responde pela associação positiva com a conversão agrícola.
   - Testamos também o lucro operacional líquido real por hectare (`luc_act_k`, `luc_full_k` e `luc_asinh`).
3. **Painel C — Moderação por Bioma (`M1b`/`M2b` com shares contínuos e subamostras Amazônia vs. Cerrado).**
4. **Painel D — Soja e Milho juntos (`M3`) e diagnóstico de colinearidade.**


In [3]:
# Painel A: Especificações principais de Receita Bruta Real/ha (Soja e Milho)
M1_act      <- run("soja",  "M1_act",      "principal", "M1 soja ativa ln(rec) c/ ef. principal", "ativa", spec = "act", include_main = TRUE)
M1_act_nolv <- run("soja",  "M1_act_nolv", "principal", "M1 soja ativa ln(rec) s/ ef. principal", "ativa", spec = "act", include_main = FALSE)
M1_1p       <- run("soja",  "M1_1p",       "principal", "M1 soja obs ln(1+rec) c/ ef. principal", "obs",   spec = "1p",  include_main = TRUE)
M1_1p_zero  <- run("soja",  "M1_1p_zero",  "principal", "M1 soja obs ln(1+rec) + dummy zero",     "obs",   spec = "1p",  include_main = TRUE, extra = c("zero_s_l1", "zero_s_zs"))
M1_lvl      <- rec(feols(y_agr ~ rs_k + xrs_k + zs_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "M1_lvl", "principal", "M1 soja obs nivel USD 1000/ha", "obs")
M1_asinh    <- rec(feols(y_agr ~ lrs_asinh + xrs_asinh + zs_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "M1_asinh", "principal", "M1 soja obs asinh(rec/ha)", "obs")

cat("=== PAINEL A.1: MODELOS PRINCIPAIS — SOJA (Receita Bruta Real/ha em t-1) ===\n")
tab(M1_act, M1_act_nolv, M1_1p, M1_1p_zero, M1_lvl, M1_asinh,
    headers = c("M1 ativa ln(rec)", "Ativa s/ princ.", "Obs ln(1+rec)", "Obs + dummy 0", "Obs USD 1k/ha", "Obs asinh"))

M2_act      <- run("milho", "M2_act",      "principal", "M2 milho ativa ln(rec) c/ ef. principal", "ativa", spec = "act", include_main = TRUE)
M2_act_nolv <- run("milho", "M2_act_nolv", "principal", "M2 milho ativa ln(rec) s/ ef. principal", "ativa", spec = "act", include_main = FALSE)
M2_1p       <- run("milho", "M2_1p",       "principal", "M2 milho obs ln(1+rec) c/ ef. principal", "obs",   spec = "1p",  include_main = TRUE)
M2_1p_zero  <- run("milho", "M2_1p_zero",  "principal", "M2 milho obs ln(1+rec) + dummy zero",     "obs",   spec = "1p",  include_main = TRUE, extra = c("zero_m_l1", "zero_m_zm"))
M2_lvl      <- rec(feols(y_agr ~ rm_k + xrm_k + zm_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "M2_lvl", "principal", "M2 milho obs nivel USD 1000/ha", "obs")
M2_asinh    <- rec(feols(y_agr ~ lrm_asinh + xrm_asinh + zm_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "M2_asinh", "principal", "M2 milho obs asinh(rec/ha)", "obs")

cat("\n=== PAINEL A.2: MODELOS PRINCIPAIS — MILHO (Receita Bruta Real/ha em t-1) ===\n")
tab(M2_act, M2_act_nolv, M2_1p, M2_1p_zero, M2_lvl, M2_asinh,
    headers = c("M2 ativa ln(rec)", "Ativa s/ princ.", "Obs ln(1+rec)", "Obs + dummy 0", "Obs USD 1k/ha", "Obs asinh"))

# Painel B: Decomposição Preço (ln P) vs. Produtividade Física PAM (ln Y_PAM) e Lucro Operacional Líquido Real/ha
D_s_prc  <- rec(feols(y_agr ~ xs + zs_tr + rain | ibge + bioma^ano, d_act_s, vcov = ~ibge), "D_prc_act_soja", "decomposicao", "soja ativa: apenas preco x Z", "ativa")
D_s_yld  <- rec(feols(y_agr ~ lyld_s_act + xyld_s_act + zs_tr + rain | ibge + bioma^ano, d_act_s, vcov = ~ibge), "D_yld_act_soja", "decomposicao", "soja ativa: apenas produtividade PAM x Z", "ativa")
D_s_both <- rec(feols(y_agr ~ xs + lyld_s_act + xyld_s_act + zs_tr + rain | ibge + bioma^ano, d_act_s, vcov = ~ibge), "D_both_act_soja", "decomposicao", "soja ativa: preco x Z + prod. PAM x Z", "ativa")
L_s_act  <- rec(feols(y_agr ~ luc_s_act_k + xluc_s_act_k + zs_tr + rain | ibge + bioma^ano, d_act_s, vcov = ~ibge), "L_act_soja", "lucro_ha", "soja ativa: lucro liquido real/ha (USD 1k)", "ativa")
L_s_full <- rec(feols(y_agr ~ luc_s_full_k + xluc_s_full_k + zs_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "L_full_soja", "lucro_ha", "soja obs: lucro liquido real/ha (USD 1k)", "obs")
L_s_ash  <- rec(feols(y_agr ~ luc_s_asinh + xluc_s_asinh + zs_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "L_asinh_soja", "lucro_ha", "soja obs: asinh(lucro liquido/ha)", "obs")

cat("\n=== PAINEL B.1: DECOMPOSIÇÃO (PREÇO vs. PRODUTIVIDADE FÍSICA PAM) E LUCRO LÍQUIDO/HA — SOJA ===\n")
tab(M1_act, D_s_prc, D_s_yld, D_s_both, L_s_act, L_s_full, L_s_ash,
    headers = c("Receita/ha (ativa)", "Só Preço (ativa)", "Só PAM (ativa)", "Preço + PAM", "Lucro/ha (ativa)", "Lucro/ha (obs)", "Lucro asinh (obs)"))

D_m_prc  <- rec(feols(y_agr ~ xm + zm_tr + rain | ibge + bioma^ano, d_act_m, vcov = ~ibge), "D_prc_act_milho", "decomposicao", "milho ativa: apenas preco x Z", "ativa")
D_m_yld  <- rec(feols(y_agr ~ lyld_m_act + xyld_m_act + zm_tr + rain | ibge + bioma^ano, d_act_m, vcov = ~ibge), "D_yld_act_milho", "decomposicao", "milho ativa: apenas produtividade PAM x Z", "ativa")
D_m_both <- rec(feols(y_agr ~ xm + lyld_m_act + xyld_m_act + zm_tr + rain | ibge + bioma^ano, d_act_m, vcov = ~ibge), "D_both_act_milho", "decomposicao", "milho ativa: preco x Z + prod. PAM x Z", "ativa")
L_m_act  <- rec(feols(y_agr ~ luc_m_act_k + xluc_m_act_k + zm_tr + rain | ibge + bioma^ano, d_act_m, vcov = ~ibge), "L_act_milho", "lucro_ha", "milho ativa: lucro liquido real/ha (USD 1k)", "ativa")
L_m_full <- rec(feols(y_agr ~ luc_m_full_k + xluc_m_full_k + zm_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "L_full_milho", "lucro_ha", "milho obs: lucro liquido real/ha (USD 1k)", "obs")
L_m_ash  <- rec(feols(y_agr ~ luc_m_asinh + xluc_m_asinh + zm_tr + rain | ibge + bioma^ano, d_obs, vcov = ~ibge), "L_asinh_milho", "lucro_ha", "milho obs: asinh(lucro liquido/ha)", "obs")

cat("\n=== PAINEL B.2: DECOMPOSIÇÃO (PREÇO vs. PRODUTIVIDADE FÍSICA PAM) E LUCRO LÍQUIDO/HA — MILHO ===\n")
tab(M2_act, D_m_prc, D_m_yld, D_m_both, L_m_act, L_m_full, L_m_ash,
    headers = c("Receita/ha (ativa)", "Só Preço (ativa)", "Só PAM (ativa)", "Preço + PAM", "Lucro/ha (ativa)", "Lucro/ha (obs)", "Lucro asinh (obs)"))

# Painel C: Moderação por bioma (shares contínuos e subamostras Amazônia vs. Cerrado)
M1b_act <- run("soja",  "M1b_act", "moderacao", "M1b soja ativa x share bioma", "ativa", spec = "act", extra = c("xrs_act_amz", "xrs_act_pan"))
M2b_act <- run("milho", "M2b_act", "moderacao", "M2b milho ativa x share bioma", "ativa", spec = "act", extra = c("xrm_act_amz", "xrm_act_pan"))
M1b_1p  <- run("soja",  "M1b_1p",  "moderacao", "M1b soja obs ln(1+rec) x share bioma", "obs", spec = "1p", extra = c("xrs_1p_amz", "xrs_1p_pan"))
M2b_1p  <- run("milho", "M2b_1p",  "moderacao", "M2b milho obs ln(1+rec) x share bioma", "obs", spec = "1p", extra = c("xrm_1p_amz", "xrm_1p_pan"))
cat("\n=== PAINEL C.1: MODERAÇÃO CONTÍNUA POR SHARE DE BIOMA (Referência = Cerrado) ===\n")
tab(M1b_act, M2b_act, M1b_1p, M2b_1p, headers = c("Soja ativa", "Milho ativa", "Soja ln(1+rec)", "Milho ln(1+rec)"))

sub_act <- list(); sub_1p <- list()
for (crop in c("soja", "milho")) {
  for (b in c(bioma_amz, bioma_cer)) {
    da <- filter(if (crop == "soja") d_act_s else d_act_m, bioma == b)
    do <- filter(d_obs, bioma == b)
    sub_act[[paste(crop, b)]] <- run(crop, paste0("S_act_", crop, "_", substr(b, 1, 3)), "subamostra",
                                     paste(crop, "ativa subamostra", b), "ativa", data = da, spec = "act", fe = "ibge + ano")
    sub_1p[[paste(crop, b)]]  <- run(crop, paste0("S_1p_", crop, "_", substr(b, 1, 3)), "subamostra",
                                     paste(crop, "obs subamostra", b), "obs", data = do, spec = "1p", fe = "ibge + ano")
  }
}
cat("\n=== PAINEL C.2: SUBAMOSTRAS POR BIOMA PREDOMINANTE — LAVOURA ATIVA (ln rec/ha) ===\n")
tab(sub_act[[1]], sub_act[[2]], sub_act[[3]], sub_act[[4]], headers = names(sub_act))
cat("\n=== PAINEL C.3: SUBAMOSTRAS POR BIOMA PREDOMINANTE — AMOSTRA COMPLETA (ln 1+rec/ha) ===\n")
tab(sub_1p[[1]], sub_1p[[2]], sub_1p[[3]], sub_1p[[4]], headers = names(sub_1p))

# Painel D: M3 (Soja e Milho juntos) e diagnóstico de colinearidade
d_act_both <- d_obs %>% filter(act_s & act_m)
M3_act <- feols(y_agr ~ lrs_act + xrs_act + lrm_act + xrm_act + zs_tr + zm_tr + rain | ibge + bioma^ano, data = d_act_both, vcov = ~ibge)
M3_1p  <- feols(y_agr ~ lrs_1p + xrs_1p + lrm_1p + xrm_1p + zs_tr + zm_tr + rain | ibge + bioma^ano, data = d_obs, vcov = ~ibge)
rec(M3_act, "M3_act", "robustez_cultura", "M3 soja + milho juntos (lavoura ativa ambos)", "ativa ambos")
rec(M3_1p,  "M3_1p",  "robustez_cultura", "M3 soja + milho juntos (obs ln 1+rec)", "obs")
cat("\n=== PAINEL D: M3 — SOJA E MILHO JUNTOS ===\n")
tab(M3_act, M3_1p, headers = c("M3 ativa (ambos > 0)", "M3 obs ln(1+rec)"))

w3_act <- wald(M3_act, keep = "^xr[sm]_act$", print = FALSE)
w3_1p  <- wald(M3_1p,  keep = "^xr[sm]_1p$",  print = FALSE)
cat(sprintf("Teste Wald conjunto (ativa, xrs_act = xrm_act = 0): F = %.3f (p = %.4f)\n", w3_act$stat, w3_act$p))
cat(sprintf("Teste Wald conjunto (obs,   xrs_1p  = xrm_1p  = 0): F = %.3f (p = %.4f)\n", w3_1p$stat,  w3_1p$p))
cat(sprintf("Correlações: lrs_act vs lrm_act = %.3f | lrs_1p vs lrm_1p = %.3f | xrs_act vs xs = %.3f | xrm_act vs xm = %.3f\n",
            cor(d_act_both$lrs_act, d_act_both$lrm_act), cor(d_obs$lrs_1p, d_obs$lrm_1p),
            cor(d_act_s$xrs_act, d_act_s$xs), cor(d_act_m$xrm_act, d_act_m$xm)))
RES[[length(RES) + 1]] <- tibble(id = "M3_act_wald", grupo = "robustez_cultura", descricao = "teste conjunto xrs_act=xrm_act=0",
  amostra = "ativa ambos", termo = "wald_xrs_xrm_act", estimativa = w3_act$stat, erro_padrao = NA_real_, p = w3_act$p, n = nobs(M3_act))
RES[[length(RES) + 1]] <- tibble(id = "M3_1p_wald", grupo = "robustez_cultura", descricao = "teste conjunto xrs_1p=xrm_1p=0",
  amostra = "obs", termo = "wald_xrs_xrm_1p", estimativa = w3_1p$stat, erro_padrao = NA_real_, p = w3_1p$p, n = nobs(M3_1p))


=== PAINEL A.1: MODELOS PRINCIPAIS — SOJA (Receita Bruta Real/ha em t-1) ===


                 M1 ativa ln(rec)     Ativa s/ princ.       Obs ln(1+rec)       Obs + dummy 0       Obs USD 1k/ha           Obs asinh
lrs_act           0.158* (0.0873)                                                                                                    
xrs_act           0.0392 (0.0311)    0.0544* (0.0277)                                                                                
zs_tr          0.0632*** (0.0216)  0.0604*** (0.0216)  0.0589*** (0.0220)   0.0343** (0.0164)   0.0515** (0.0204)  0.0590*** (0.0220)
rain          -0.0152** (0.00638) -0.0158** (0.00643) -0.0124** (0.00581) -0.0131** (0.00600) -0.0131** (0.00616) -0.0124** (0.00581)
lrs_1p                                                 0.0109** (0.00429)   -0.204** (0.0826)                                        
xrs_1p                                                 -0.00338 (0.00306)    0.128** (0.0503)                                        
zero_s_l1                                                     


=== PAINEL A.2: MODELOS PRINCIPAIS — MILHO (Receita Bruta Real/ha em t-1) ===


                 M2 ativa ln(rec)     Ativa s/ princ.       Obs ln(1+rec)       Obs + dummy 0       Obs USD 1k/ha           Obs asinh
lrm_act          -0.0606 (0.0437)                                                                                                    
xrm_act          0.0306* (0.0170)    0.0298* (0.0172)                                                                                
zm_tr          0.0511*** (0.0189)  0.0509*** (0.0189)  0.0620*** (0.0178)  0.0472*** (0.0179)  0.0416*** (0.0151)  0.0623*** (0.0179)
rain          -0.0149** (0.00669) -0.0150** (0.00669) -0.0150** (0.00667) -0.0149** (0.00666) -0.0146** (0.00658) -0.0150** (0.00667)
lrm_1p                                                 -0.00143 (0.00343)   -0.0726* (0.0394)                                        
xrm_1p                                                0.00589** (0.00240)   0.0351** (0.0164)                                        
zero_m_l1                                                     


=== PAINEL B.1: DECOMPOSIÇÃO (PREÇO vs. PRODUTIVIDADE FÍSICA PAM) E LUCRO LÍQUIDO/HA — SOJA ===


               Receita/ha (ativa)    Só Preço (ativa)      Só PAM (ativa)         Preço + PAM     Lucro/ha (ativa)      Lucro/ha (obs)
lrs_act           0.158* (0.0873)                                                                                                     
xrs_act           0.0392 (0.0311)                                                                                                     
zs_tr          0.0632*** (0.0216)   0.0513** (0.0198)  0.0988*** (0.0315)  0.0751*** (0.0242)   0.0806*** (0.0270)  0.0553*** (0.0207)
rain          -0.0152** (0.00638) -0.0168** (0.00668) -0.0143** (0.00619) -0.0160** (0.00659) -0.00903** (0.00448) -0.00719* (0.00386)
xs                                  0.143*** (0.0526)                        0.132** (0.0506)                                         
lyld_s_act                                              0.253*** (0.0873)   0.238*** (0.0861)                                         
xyld_s_act                                             

... (9 outputs left out)

              M3 ativa (ambos > 0)    M3 obs ln(1+rec)
lrs_act           0.175** (0.0863)                    
xrs_act            0.0259 (0.0304)                    
lrm_act           -0.0685 (0.0558)                    
xrm_act            0.0355 (0.0219)                    
zs_tr              0.0481 (0.0291)     0.0446 (0.0276)
zm_tr              0.0281 (0.0380)     0.0315 (0.0255)
rain           -0.0156** (0.00632) -0.0124** (0.00575)
lrs_1p                              0.0107** (0.00433)
xrs_1p                              -0.00403 (0.00289)
lrm_1p                              -0.00342 (0.00384)
xrm_1p                             0.00748** (0.00367)
N                             2066                2767
R2 within                   0.0288              0.0250
Efeitos fixos     ibge + bioma^ano    ibge + bioma^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01


Teste Wald conjunto (ativa, xrs_act = xrm_act = 0): F = 1.938 (p = 0.1482)


Teste Wald conjunto (obs,   xrs_1p  = xrm_1p  = 0): F = 2.557 (p = 0.0814)


Correlações: lrs_act vs lrm_act = 0.814 | lrs_1p vs lrm_1p = 0.333 | xrs_act vs xs = 0.886 | xrm_act vs xm = 0.642


***
##### 2.4 Bateria de Robustez: Inferência, Placebo, Janelas, Desfecho, Custos e Amostra Fiscal (Etapas 7 a 11)

Avaliamos toda a bateria de robustez tanto para a especificação na **amostra com lavoura ativa (`spec = 'act'`, `ln(rec/ha)`)** quanto na **amostra observada completa (`spec = '1p'`, `ln(1+rec/ha)`)**:
- **Inferência (Etapa 7):** cluster duplo (`ibge + ano`), cluster por microrregião (22 clusters), FE `ibge + micro^ano`, regressão em 24 pontos anuais (Newey-West HAC lag 2) e *leave-one-year-out* (24 subamostras).
- **Placebo e janelas (Etapa 8):** rendimento futuro ($\ln(\text{rec/ha}_{t+1}) \times Z_i$) e recorte pós-ruptura (2006–2024).
- **Desfecho e amostra (Etapa 9):** winsorização a 99%, $\text{asinh}(10Y)$, exclusão do Pantanal, amostra preenchida (3.384 obs) e conversão para pastagem (`y_pas`).
- **Custos na base (Etapa 11) e Autos na amostra fiscal (Etapa 10):** inclusão de $\ln\text{DAP}_{t-1}\times Z$ e $\ln\text{CONAB}_{t-1}\times Z$, e estimação de M6/M7 nos pares município-ano elegíveis da amostra fiscal.


In [4]:
# Etapa 7: Inferência alternativa (Amostra Ativa e Amostra Completa ln(1+rec))
for (sp in c("act", "1p")) {
  am_lbl <- if (sp == "act") "ativa" else "obs"
  for (crop in c("soja", "milho")) {
    r1 <- run(crop, paste0("R_cl2_", sp, "_", crop), "robustez_inferencia", paste(crop, sp, "cluster municipio+ano"), am_lbl, spec = sp, vcov = ~ibge + ano)
    r2 <- run(crop, paste0("R_clmic_", sp, "_", crop), "robustez_inferencia", paste(crop, sp, "cluster microrregiao"), am_lbl, spec = sp, vcov = ~micro)
    r3 <- run(crop, paste0("R_micano_", sp, "_", crop), "robustez_inferencia", paste(crop, sp, "FE microrregiao x ano"), am_lbl, spec = sp, fe = "ibge + micro^ano")
    cat(sprintf("=== Inferência alternativa (%s): %s ===\n", toupper(sp), toupper(crop)))
    tab(r1, r2, r3, headers = paste(crop, sp, c("cluster mun+ano", "cluster micro", "FE micro x ano")))
  }
}

# Regressão em 24 pontos anuais (Newey-West HAC lag 2): inclinação anual Y ~ Z contra a média anual de ln(rec/ha)
two_step_r <- function(crop, sp = "act") {
  d <- if (sp == "act") (if (crop == "soja") d_act_s else d_act_m) else d_obs
  d <- d %>% group_by(ibge) %>% mutate(ydm = y_agr - mean(y_agr)) %>% ungroup()
  d$zv  <- if (crop == "soja") d$zs else d$zm
  d$lrv <- if (sp == "act") (if (crop == "soja") d$lrs_act else d$lrm_act) else (if (crop == "soja") d$lrs_1p else d$lrm_1p)
  sl <- d %>% group_by(ano) %>% summarise(slope = cov(ydm, zv) / var(zv), lr = mean(lrv, na.rm = TRUE), .groups = "drop")
  m <- lm(slope ~ lr, data = sl)
  ct <- coeftest(m, vcov. = NeweyWest(m, lag = 2, prewhite = FALSE))
  RES[[length(RES) + 1]] <<- tibble(id = paste0("R_2step_", sp, "_", crop), grupo = "robustez_inferencia",
    descricao = paste(crop, sp, "regressao em 24 pontos (HAC lag 2)"), amostra = ifelse(sp == "act", "ativa", "obs"),
    termo = "lr", estimativa = ct["lr", 1], erro_padrao = ct["lr", 2], p = ct["lr", 4], n = nrow(sl))
  cat(sprintf("Regressão 24 pontos (%s, %s): coef = %.4f (EP HAC = %.4f, p = %.4f)\n", crop, sp, ct["lr", 1], ct["lr", 2], ct["lr", 4]))
}
loyo_r <- function(crop, sp = "act") {
  d <- if (sp == "act") (if (crop == "soja") d_act_s else d_act_m) else d_obs
  x <- if (sp == "act") (if (crop == "soja") "xrs_act" else "xrm_act") else (if (crop == "soja") "xrs_1p" else "xrm_1p")
  out <- map_dfr(sort(unique(d$ano)), function(a) {
    m <- fit_run(crop, data = filter(d, ano != a), spec = sp)
    rec(m, paste0("R_loyo_", sp, "_", crop, "_", a), "robustez_loyo", paste(crop, sp, "exclui ano", a), ifelse(sp == "act", "ativa", "obs"))
    ct <- coeftable(m)
    tibble(ano_excluido = a, estimativa = ct[x, 1], p = ct[x, 4])
  })
  cat(sprintf("Leave-one-year-out (%s, %s): min = %.4f, max = %.4f | positivos = %d/24 | p < 0,10 = %d/24\n",
              crop, sp, min(out$estimativa), max(out$estimativa), sum(out$estimativa > 0), sum(out$p < 0.10)))
}
for (sp in c("act", "1p")) {
  two_step_r("soja", sp); two_step_r("milho", sp)
  loyo_r("soja", sp); loyo_r("milho", sp)
}


=== Inferência alternativa (ACT): SOJA ===
              soja act cluster mun+ano soja act cluster micro soja act FE micro x ano
lrs_act                0.158* (0.0891)          0.158 (0.101)         -0.0540 (0.114)
xrs_act                0.0392 (0.0530)        0.0392 (0.0373)         0.0734 (0.0551)
zs_tr                0.0632** (0.0249)       0.0632* (0.0334)         0.0460 (0.0413)
rain                 -0.0152 (0.00908)     -0.0152* (0.00763)        -0.0112 (0.0109)
N                                 2091                   2091                    2059
R2 within                       0.0247                 0.0247                  0.0088
Efeitos fixos         ibge + bioma^ano       ibge + bioma^ano        ibge + micro^ano
Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01
=== Inferência alternativa (ACT): MILHO ===
              milho act cluster mun+ano milho act cluster micro milho act FE micro x ano
lrm_act                -0.0606 (0.0411

Regressão 24 pontos (soja, act): coef = 0.1918 (EP HAC = 0.0954, p = 0.0569)
Regressão 24 pontos (milho, act): coef = 0.0806 (EP HAC = 0.0321, p = 0.0198)
Leave-one-year-out (soja, act): min = 0.0063, max = 0.0671 | positivos = 24/24 | p < 0,10 = 2/24
Leave-one-year-out (milho, act): min = 0.0232, max = 0.0418 | positivos = 24/24 | p < 0,10 = 21/24
Regressão 24 pontos (soja, 1p): coef = 0.0325 (EP HAC = 0.0154, p = 0.0461)
Regressão 24 pontos (milho, 1p): coef = 0.0757 (EP HAC = 0.0298, p = 0.0185)
Leave-one-year-out (soja, 1p): min = -0.0040, max = -0.0013 | positivos = 0/24 | p < 0,10 = 0/24
Leave-one-year-out (milho, 1p): min = 0.0038, max = 0.0085 | positivos = 24/24 | p < 0,10 = 24/24


In [5]:
# Etapas 8 e 9: Placebo (receita/ha em t+1), janelas 2006-2024 e transformações de Y / amostra
q99 <- quantile(d_obs$y_agr, 0.99)
df_y <- df_r1 %>% mutate(y_w = pmin(y_agr, q99), y_a = asinh(10 * y_agr))

for (sp in c("act", "1p")) {
  am_lbl <- if (sp == "act") "ativa" else "obs"
  for (crop in c("soja", "milho")) {
    d_base <- if (sp == "act") filter(df_y, obs & (if (crop == "soja") act_s else act_m)) else filter(df_y, obs)
    d_full <- if (sp == "act") filter(df_y, if (crop == "soja") act_s else act_m) else df_y
    xf <- if (sp == "act") { if (crop == "soja") c("lrs_act_f", "xrs_act_f") else c("lrm_act_f", "xrm_act_f") }
          else             { if (crop == "soja") c("lrs_1p_f",  "xrs_1p_f")  else c("lrm_1p_f",  "xrm_1p_f") }
    p1 <- run(crop, paste0("R_plac_", sp, "_", crop), "placebo", paste(crop, sp, "placebo receita t+1"), am_lbl, data = d_base, spec = sp, extra = xf)
    w1 <- run(crop, paste0("R_w06_", sp, "_", crop), "janela", paste(crop, sp, "2006-2024 c/ tend."), paste(am_lbl, "2006-2024"), data = filter(d_base, ano >= 2006), spec = sp)
    w2 <- run(crop, paste0("R_w06nt_", sp, "_", crop), "janela", paste(crop, sp, "2006-2024 s/ tend."), paste(am_lbl, "2006-2024"), data = filter(d_base, ano >= 2006), spec = sp, trend = FALSE)
    w3 <- run(crop, paste0("R_nt_", sp, "_", crop), "janela", paste(crop, sp, "2001-2024 s/ tend."), am_lbl, data = d_base, spec = sp, trend = FALSE)
    cat(sprintf("=== Placebo e Janelas (%s): %s ===\n", toupper(sp), toupper(crop)))
    tab(p1, w1, w2, w3, headers = paste(crop, sp, c("placebo t+1", "2006-24 c/ tend.", "2006-24 s/ tend.", "2001-24 s/ tend.")))

    a <- run(crop, paste0("R_win_", sp, "_", crop), "robustez_y", paste(crop, sp, "Y winsor 99%"), am_lbl, data = d_base, spec = sp, y = "y_w")
    b <- run(crop, paste0("R_asinh_", sp, "_", crop), "robustez_y", paste(crop, sp, "asinh(10Y)"), am_lbl, data = d_base, spec = sp, y = "y_a")
    c <- run(crop, paste0("R_nopan_", sp, "_", crop), "robustez_amostra", paste(crop, sp, "sem Pantanal"), paste(am_lbl, "sem Pantanal"), data = filter(d_base, bioma != bioma_pan), spec = sp)
    d <- run(crop, paste0("R_full_", sp, "_", crop), "robustez_amostra", paste(crop, sp, "Y preenchida"), "todas", data = d_full, spec = sp)
    e <- run(crop, paste0("R_pas_", sp, "_", crop), "pastagem", paste(crop, sp, "Y = pastagem"), am_lbl, data = d_base, spec = sp, y = "y_pas")
    cat(sprintf("=== Desfecho e Amostra (%s): %s ===\n", toupper(sp), toupper(crop)))
    tab(a, b, c, d, e, headers = paste(crop, sp, c("winsor 99%", "asinh", "sem Pantanal", "Y preenchida", "Y pastagem")))
  }
}


=== Placebo e Janelas (ACT): SOJA ===
              soja act placebo t+1 soja act 2006-24 c/ tend. soja act 2006-24 s/ tend. soja act 2001-24 s/ tend.
lrs_act            0.0818 (0.0769)         0.0456** (0.0223)          0.0459* (0.0235)            0.132 (0.0982)
xrs_act            0.0363 (0.0279)          -0.0118 (0.0106)         0.000908 (0.0121)          0.128** (0.0517)
zs_tr           0.0642*** (0.0211)        0.0176** (0.00807)                                                    
rain           -0.0121** (0.00612)        -0.00181 (0.00179)        -0.00273 (0.00189)      -0.0192*** (0.00695)
lrs_act_f          -0.0274 (0.148)                                                                              
xrs_act_f         0.120** (0.0480)                                                                              
N                             1897                      1732                      1732                      2091
R2 within                   0.0308                    0.00

In [6]:
# Etapa 11: Inclusão de custos (DAP e CONAB) como controles de margem
for (crop in c("soja", "milho")) {
  xd <- if (crop == "soja") "x_dap_s" else "x_dap_m"
  xc <- if (crop == "soja") "x_cst_s" else "x_cst_m"
  c1_a <- run(crop, paste0("C_dap_act_", crop),  "custos", paste(crop, "ativa + DAP x Z"),   "ativa", spec = "act", extra = xd)
  c2_a <- run(crop, paste0("C_con_act_", crop),  "custos", paste(crop, "ativa + CONAB x Z"), "ativa", spec = "act", extra = xc)
  c3_a <- run(crop, paste0("C_both_act_", crop), "custos", paste(crop, "ativa + DAP + CONAB"), "ativa", spec = "act", extra = c(xd, xc))
  c1_o <- run(crop, paste0("C_dap_1p_", crop),   "custos", paste(crop, "obs + DAP x Z"),     "obs",   spec = "1p",  extra = xd)
  c2_o <- run(crop, paste0("C_con_1p_", crop),   "custos", paste(crop, "obs + CONAB x Z"),   "obs",   spec = "1p",  extra = xc)
  cat("=== Custos da base (DAP e CONAB):", toupper(crop), "===\n")
  tab(c1_a, c2_a, c3_a, c1_o, c2_o, headers = paste(crop, c("ativa+DAP", "ativa+CONAB", "ativa+ambos", "obs+DAP", "obs+CONAB")))
}

# Etapa 10: Subamostra fiscal elegível (fisc = TRUE)
d_fisc_act_s <- d_fisc %>% filter(act_s)
d_fisc_act_m <- d_fisc %>% filter(act_m)
M6_act_s <- run("soja",  "M6_act_soja",  "fiscal", "M6 soja ativa + autos", "fiscal ativa", data = d_fisc_act_s, spec = "act", trend = FALSE, extra = "f1")
M7_act_s <- run("soja",  "M7_act_soja",  "fiscal", "M7 soja ativa x autos", "fiscal ativa", data = d_fisc_act_s, spec = "act", trend = FALSE, extra = c("f1", "f1_rs_act", "f1_zs", "f1_xrs_act"))
M6_act_m <- run("milho", "M6_act_milho", "fiscal", "M6 milho ativa + autos", "fiscal ativa", data = d_fisc_act_m, spec = "act", trend = FALSE, extra = "f1")
M7_act_m <- run("milho", "M7_act_milho", "fiscal", "M7 milho ativa x autos", "fiscal ativa", data = d_fisc_act_m, spec = "act", trend = FALSE, extra = c("f1", "f1_rm_act", "f1_zm", "f1_xrm_act"))
M7_1p_s  <- run("soja",  "M7_1p_soja",   "fiscal", "M7 soja obs ln(1+rec) x autos", "fiscal", data = d_fisc, spec = "1p", trend = FALSE, extra = c("f1", "f1_rs_1p", "f1_zs", "f1_xrs_1p"))
M7_1p_m  <- run("milho", "M7_1p_milho",  "fiscal", "M7 milho obs ln(1+rec) x autos", "fiscal", data = d_fisc, spec = "1p", trend = FALSE, extra = c("f1", "f1_rm_1p", "f1_zm", "f1_xrm_1p"))
cat("\n=== Autos de Flora na Amostra Fiscal Elegível (fisc = TRUE) ===\n")
tab(M6_act_s, M7_act_s, M6_act_m, M7_act_m, M7_1p_s, M7_1p_m,
    headers = c("M6 soja (ativa)", "M7 soja (ativa)", "M6 milho (ativa)", "M7 milho (ativa)", "M7 soja (obs)", "M7 milho (obs)"))


=== Custos da base (DAP e CONAB): SOJA ===
                   soja ativa+DAP    soja ativa+CONAB    soja ativa+ambos        soja obs+DAP      soja obs+CONAB
lrs_act         0.219*** (0.0810)    0.160** (0.0757)   0.208*** (0.0755)                                        
xrs_act        -0.141*** (0.0432)    -0.0149 (0.0221)  -0.156*** (0.0460)                                        
zs_tr          0.0627*** (0.0214)   0.0393** (0.0169)   0.0426** (0.0179)  0.0413*** (0.0153)   0.0345** (0.0134)
rain          -0.0165** (0.00663) -0.0101** (0.00467) -0.0114** (0.00491) -0.0134** (0.00602) -0.00730* (0.00374)
x_dap_s         0.173*** (0.0474)                       0.141*** (0.0389)   0.0610** (0.0242)                    
x_cst_s                             0.207*** (0.0551)   0.189*** (0.0505)                       0.127*** (0.0470)
lrs_1p                                                                     0.0105** (0.00424) 0.00895** (0.00380)
xrs_1p                                       


=== Autos de Flora na Amostra Fiscal Elegível (fisc = TRUE) ===


                   M6 soja (ativa)      M7 soja (ativa)    M6 milho (ativa)    M7 milho (ativa)       M7 soja (obs)      M7 milho (obs)
lrs_act           0.208** (0.0961)        0.153 (0.114)                                                                                
xrs_act           0.115** (0.0575)      0.117* (0.0687)                                                                                
rain          -0.00833** (0.00408) -0.00883** (0.00418) -0.00642* (0.00360) -0.00653* (0.00372) -0.00602* (0.00336) -0.00676* (0.00350)
f1               0.00343 (0.00478)    0.00600 (0.00557)   0.00309 (0.00643)   0.00118 (0.00719)   0.00564 (0.00562)   0.00231 (0.00544)
f1_rs_act                               0.0433 (0.0262)                                                                                
f1_zs                                   0.0217 (0.0137)                                           0.00615 (0.00920)                    
f1_xrs_act                              0.0106 (

***
##### 2.5 CAMINHO A — Quebra Estrutural (2005/2006), Cenários Alternativos de Capacidade e Ocupação Pré-Determinada (1985–2000) (Etapas 14B e 14C)

Repetimos os testes do Caminho A substituindo o choque de preços reais pelo rendimento real por hectare (`lrs_act`/`lrs_1p` e `lrm_act`/`lrm_1p`):
1. **Quebra estrutural de regime (`post05` e `post06`)** interagida com $Z_i$ e `amz`;
2. **Cenários alternativos de capacidade produtiva** (`gaez_yxLr0_soy`, `gaez_sxHr0_soy`, `pam_1991_2000_min7`);
3. **Exposição agrícola pré-determinada (MapBiomas 1985–2000 e soja em 2000)** e **intensificação sobre a área plantada de milho e soja (% da área municipal)**.


In [7]:
# A.1 Quebra estrutural de regime (pós-2005 e pós-2006)
ev_r <- function(crop, pcol, sp = "act") {
  z  <- if (crop == "soja") "zs" else "zm"
  lv <- if (sp == "act") (if (crop == "soja") "lrs_act" else "lrm_act") else (if (crop == "soja") "lrs_1p" else "lrm_1p")
  x  <- if (sp == "act") (if (crop == "soja") "xrs_act" else "xrm_act") else (if (crop == "soja") "xrs_1p" else "xrm_1p")
  d0 <- if (sp == "act") (if (crop == "soja") d_act_s else d_act_m) else d_obs
  d  <- d0 %>% mutate(p = .data[[pcol]], e_amz = amz * p, e_z = .data[[z]] * p, e_zamz = .data[[z]] * amz * p)
  m  <- feols(as.formula(paste("y_agr ~", lv, "+", x, "+ e_amz + e_z + e_zamz + rain | ibge + ano")), d, vcov = ~ibge)
  rec(m, paste0("E_", sp, "_", pcol, "_", crop), "eventos", paste(crop, sp, "evento", pcol), ifelse(sp == "act", "ativa", "obs"))
}
cat("=== CAMINHO A.1: Quebra estrutural pós-2005 e pós-2006 (Lavoura Ativa e Amostra Completa) ===\n")
tab(ev_r("soja", "post06", "act"), ev_r("milho", "post06", "act"), ev_r("soja", "post06", "1p"), ev_r("milho", "post06", "1p"),
    headers = c("Soja ativa pós-06", "Milho ativa pós-06", "Soja obs pós-06", "Milho obs pós-06"))

# A.2 Cenários alternativos de capacidade (GAEZ alto vs. baixo insumo, índice sx e PAM 1991-2000)
x1_path <- file.path(root, "xdata/chap1/processed/harmonizacao/x1_psoja-pmilho/exposicoes_alternativas.csv")
x1_alt <- fread(x1_path, showProgress = FALSE) %>%
  as_tibble() %>%
  select(ibge_int = codigo_ibge, ano, produto, cenario_capacidade, capacidade_z) %>%
  distinct() %>%
  mutate(key = paste0(cenario_capacidade, "_", produto)) %>%
  select(ibge_int, ano, key, capacidade_z) %>%
  pivot_wider(names_from = key, values_from = capacidade_z)

# A.3 Exposição pré-determinada MapBiomas 1985-2000
mb_path <- file.path(root, "xdata/chap1/processed/tabular/h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv")
mb_pre <- fread(mb_path, showProgress = FALSE) %>%
  as_tibble() %>%
  filter(ano >= 1985, ano <= 2000) %>%
  group_by(ibge_int = codigo_ibge, ano) %>%
  summarise(
    tot_ha  = sum(area_ha),
    agr_ha  = sum(area_ha[class_id %in% c(20, 39, 41, 62)]),
    soja_ha = sum(area_ha[class_id == 39]),
    pas_ha  = sum(area_ha[class_id == 15]),
    .groups = "drop"
  ) %>%
  group_by(ibge_int) %>%
  summarise(
    area_mun_ha  = first(tot_ha),
    agr_sh_85_00 = mean(agr_ha / tot_ha),
    pas_sh_85_00 = mean(pas_ha / tot_ha),
    soja_sh_2000 = (soja_ha / tot_ha)[ano == 2000],
    .groups = "drop"
  ) %>%
  mutate(
    z_exp_agr  = (agr_sh_85_00 - mean(agr_sh_85_00)) / sd_pop(agr_sh_85_00),
    z_exp_pas  = (pas_sh_85_00 - mean(pas_sh_85_00)) / sd_pop(pas_sh_85_00),
    z_exp_soja = (soja_sh_2000 - mean(soja_sh_2000)) / sd_pop(soja_sh_2000)
  )

d_camA <- d_obs %>%
  left_join(x1_alt, by = c("ibge_int", "ano")) %>%
  left_join(mb_pre, by = "ibge_int") %>%
  mutate(
    # Cenários alternativos (soja ativa e obs)
    xrs_act_yxLr  = lrs_act * gaez_yxLr0_soy_soja,  zs_yxLr_tr  = gaez_yxLr0_soy_soja * trend,
    xrs_act_sxHr  = lrs_act * gaez_sxHr0_soy_soja,  zs_sxHr_tr  = gaez_sxHr0_soy_soja * trend,
    xrs_act_pam90 = lrs_act * pam_1991_2000_min7_soja, zs_pam90_tr = pam_1991_2000_min7_soja * trend,
    xrs_1p_yxLr   = lrs_1p  * gaez_yxLr0_soy_soja,
    xrs_1p_sxHr   = lrs_1p  * gaez_sxHr0_soy_soja,
    xrs_1p_pam90  = lrs_1p  * pam_1991_2000_min7_soja,
    # Ocupação agrícola pré-determinada (1985-2000)
    x_exp_agr_rs_act  = lrs_act * z_exp_agr,  z_exp_agr_tr  = z_exp_agr * trend,
    x_exp_soja_rs_act = lrs_act * z_exp_soja, z_exp_soja_tr = z_exp_soja * trend,
    x_exp_agr_rm_act  = lrm_act * z_exp_agr,
    x_exp_agr_rs_1p   = lrs_1p  * z_exp_agr,
    x_exp_agr_rm_1p   = lrm_1p  * z_exp_agr,
    soja_plant_pct    = 100 * area_soja_ha / area_mun_ha,
    milho_plant_pct   = 100 * area_milho_ha / area_mun_ha
  )

d_camA_act_s <- d_camA %>% filter(act_s)
d_camA_act_m <- d_camA %>% filter(act_m)

CA_yxHr <- M1_act
CA_yxLr <- rec(feols(y_agr ~ lrs_act + xrs_act_yxLr + zs_yxLr_tr + rain | ibge + bioma^ano, d_camA_act_s, vcov = ~ibge), "CA_act_yxLr_soja", "caminho_A", "soja ativa GAEZ baixo insumo yxLr0", "ativa")
CA_sxHr <- rec(feols(y_agr ~ lrs_act + xrs_act_sxHr + zs_sxHr_tr + rain | ibge + bioma^ano, d_camA_act_s, vcov = ~ibge), "CA_act_sxHr_soja", "caminho_A", "soja ativa GAEZ indice aptidao sxHr0", "ativa")
CA_pam  <- rec(feols(y_agr ~ lrs_act + xrs_act_pam90 + zs_pam90_tr + rain | ibge + bioma^ano, d_camA_act_s, vcov = ~ibge), "CA_act_pam90_soja", "caminho_A", "soja ativa PAM 1991-2000", "ativa PAM>=7a")
cat("\n=== CAMINHO A.2: Comparação entre cenários de capacidade — SOJA (Lavoura Ativa) ===\n")
tab(CA_yxHr, CA_yxLr, CA_sxHr, CA_pam, headers = c("GAEZ yx alto (M1)", "GAEZ yx baixo", "GAEZ sx alto", "PAM 1991-2000"))

EX1 <- rec(feols(y_agr ~ lrs_act + x_exp_agr_rs_act + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA_act_s, vcov = ~ibge), "EX_act_agr_soja", "exposicao_pre", "soja ativa: rec x share agr 1985-2000", "ativa")
EX2 <- rec(feols(y_agr ~ lrs_act + xrs_act + x_exp_agr_rs_act + zs_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA_act_s, vcov = ~ibge), "EX_act_ambos_soja", "exposicao_pre", "soja ativa: GAEZ + share agr 1985-2000", "ativa")
EX3 <- rec(feols(y_agr ~ lrm_act + xrm_act + x_exp_agr_rm_act + zm_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA_act_m, vcov = ~ibge), "EX_act_ambos_milho", "exposicao_pre", "milho ativa: GAEZ + share agr 1985-2000", "ativa")
EX4 <- rec(feols(soja_plant_pct  ~ lrs_act + xrs_act + x_exp_agr_rs_act + zs_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA_act_s, vcov = ~ibge), "EX_act_intens_soja", "exposicao_pre", "Y = % area plantada soja (ativa)", "ativa")
EX5 <- rec(feols(milho_plant_pct ~ lrm_act + xrm_act + x_exp_agr_rm_act + zm_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA_act_m, vcov = ~ibge), "EX_act_intens_milho", "exposicao_pre", "Y = % area plantada milho (ativa)", "ativa")
cat("\n=== CAMINHO A.3: Potencial GAEZ vs. Fronteira Consolidada (1985-2000) e Intensificação ===\n")
tab(EX1, EX2, EX3, EX4, EX5, headers = c("Y=conv: agr 85-00", "Y=conv: GAEZ+agr (S)", "Y=conv: GAEZ+agr (M)", "Y=% área soja", "Y=% área milho"))


=== CAMINHO A.1: Quebra estrutural pós-2005 e pós-2006 (Lavoura Ativa e Amostra Completa) ===


                Soja ativa pós-06  Milho ativa pós-06     Soja obs pós-06    Milho obs pós-06
lrs_act           0.137* (0.0717)                                                            
xrs_act         -0.0308* (0.0158)                                                            
e_amz               0.121 (0.172)     -0.0956 (0.109)     -0.0246 (0.104)      -0.100 (0.108)
e_z              0.122** (0.0613)   0.116*** (0.0361)    0.110** (0.0498)   0.108*** (0.0324)
e_zamz             0.501* (0.301)      0.0466 (0.144)      0.0300 (0.110)      0.0360 (0.140)
rain          -0.00909* (0.00548) -0.0116** (0.00565)  -0.0100* (0.00525) -0.0117** (0.00569)
lrm_act                              -0.0443 (0.0425)                                        
xrm_act                            -0.00367 (0.00837)                                        
lrs_1p                                                0.0102*** (0.00376)                    
xrs_1p                                                 -0.00


=== CAMINHO A.2: Comparação entre cenários de capacidade — SOJA (Lavoura Ativa) ===


                GAEZ yx alto (M1)        GAEZ yx baixo         GAEZ sx alto       PAM 1991-2000
lrs_act           0.158* (0.0873)     0.207** (0.0853)     0.207** (0.0854)       0.102 (0.105)
xrs_act           0.0392 (0.0311)                                                              
zs_tr          0.0632*** (0.0216)                                                              
rain          -0.0152** (0.00638) -0.0199*** (0.00698) -0.0173*** (0.00622) -0.0283*** (0.0105)
xrs_act_yxLr                          -0.0315 (0.0310)                                         
zs_yxLr_tr                            0.00108 (0.0251)                                         
xrs_act_sxHr                                             -0.0612** (0.0262)                    
zs_sxHr_tr                                              -0.0738*** (0.0273)                    
xrs_act_pam90                                                                  -0.0201 (0.0297)
zs_pam90_tr                             


=== CAMINHO A.3: Potencial GAEZ vs. Fronteira Consolidada (1985-2000) e Intensificação ===


                    Y=conv: agr 85-00 Y=conv: GAEZ+agr (S) Y=conv: GAEZ+agr (M)    Y=% área soja   Y=% área milho
lrs_act                0.103 (0.0879)      0.0870 (0.0887)                         -2.43* (1.33)                 
x_exp_agr_rs_act     -0.0584 (0.0356)     -0.0546 (0.0331)                         0.517 (0.776)                 
z_exp_agr_tr      -0.0661*** (0.0217)   -0.0586** (0.0242)    -0.0436* (0.0256)   0.0704 (0.482)  2.15*** (0.561)
rain             -0.0177*** (0.00630)  -0.0155** (0.00621)  -0.0141** (0.00604)  0.220** (0.104) 0.442*** (0.140)
xrs_act                                    0.0278 (0.0279)                        -0.134 (0.545)                 
zs_tr                                      0.0320 (0.0247)                        -0.589 (0.616)                 
lrm_act                                                       -0.0732* (0.0408)                     0.439 (0.482)
xrm_act                                                      -0.000324 (0.0135)         

***
##### 2.6 CAMINHO B1 — Logística, Distância Geodésica aos Portos e Choques de Infraestrutura (Etapa 14A)

Avaliamos a moderação pela proximidade aos portos do Sul (`prox_sul_z`), a distância mínima efetiva aos portos ativos (`dist_min_t_100km`, com entrada de Miritituba/Arco Norte em 2014) e o choque da Ferrovia de Rondonópolis (2013) nos modelos de rendimentos reais por hectare.


In [8]:
zip_path <- normalizePath(file.path(root, "xdata/chap1/raw/vector/ibge/MT_Municipios_2023.zip"), winslash = "/")
shp_mt <- st_read(paste0("/vsizip/", zip_path), quiet = TRUE)
stopifnot(nrow(shp_mt) == 141)
cent_mt <- st_centroid(st_geometry(shp_mt))

pts_log <- st_sfc(
  st_point(c(-46.3042, -23.9619)), # Porto de Santos (SP)
  st_point(c(-48.5106, -25.5047)), # Porto de Paranaguá (PR)
  st_point(c(-55.9750,  -4.2690)), # Miritituba / Itaituba (PA - Arco Norte)
  st_point(c(-54.7167, -16.5833)), # Terminal Ferroviário de Rondonópolis (MT - 2013)
  crs = st_crs(shp_mt)
)
dmat <- as.numeric(st_distance(cent_mt, pts_log)) / 1000
dim(dmat) <- c(nrow(shp_mt), 4)

dist_mun <- tibble(
  ibge_int      = as.integer(shp_mt$CD_MUN),
  dist_sul_km   = pmin(dmat[, 1], dmat[, 2]),
  dist_mirit_km = dmat[, 3],
  dist_rond_km  = dmat[, 4]
) %>%
  mutate(
    prox_sul_z  = -(dist_sul_km - mean(dist_sul_km)) / sd_pop(dist_sul_km),
    prox_rond_z = -(dist_rond_km - mean(dist_rond_km)) / sd_pop(dist_rond_km)
  )

d_camB <- d_obs %>%
  left_join(dist_mun, by = "ibge_int") %>%
  mutate(
    dist_min_t_100km = ifelse(ano <= 2013, dist_sul_km, pmin(dist_sul_km, dist_mirit_km)) / 100,
    x_rs_act_proxsul = lrs_act * prox_sul_z,
    xrs_act_proxsul  = xrs_act * prox_sul_z,
    x_rm_act_proxsul = lrm_act * prox_sul_z,
    xrm_act_proxsul  = xrm_act * prox_sul_z,
    x_rs_1p_proxsul  = lrs_1p  * prox_sul_z,
    xrs_1p_proxsul   = xrs_1p  * prox_sul_z,
    x_rm_1p_proxsul  = lrm_1p  * prox_sul_z,
    xrm_1p_proxsul   = xrm_1p  * prox_sul_z,
    choc_ferrov_13   = prox_rond_z * as.numeric(ano >= 2013)
  )
d_camB_act_s <- d_camB %>% filter(act_s)
d_camB_act_m <- d_camB %>% filter(act_m)

L1_act_s <- rec(feols(y_agr ~ lrs_act + xrs_act + xrs_act_proxsul + x_rs_act_proxsul + zs_tr + rain | ibge + bioma^ano, d_camB_act_s, vcov = ~ibge), "B1_act_prox_soja", "caminho_B_logistica", "soja ativa x proximidade porto Sul", "ativa")
L2_act_s <- rec(feols(y_agr ~ lrs_act + xrs_act + dist_min_t_100km + choc_ferrov_13 + zs_tr + rain | ibge + bioma^ano, d_camB_act_s, vcov = ~ibge), "B1_act_choq_soja", "caminho_B_logistica", "soja ativa + Arco Norte 2014 e Ferrovia 2013", "ativa")
L1_act_m <- rec(feols(y_agr ~ lrm_act + xrm_act + xrm_act_proxsul + x_rm_act_proxsul + zm_tr + rain | ibge + bioma^ano, d_camB_act_m, vcov = ~ibge), "B1_act_prox_milho", "caminho_B_logistica", "milho ativa x proximidade porto Sul", "ativa")
L2_act_m <- rec(feols(y_agr ~ lrm_act + xrm_act + dist_min_t_100km + choc_ferrov_13 + zm_tr + rain | ibge + bioma^ano, d_camB_act_m, vcov = ~ibge), "B1_act_choq_milho", "caminho_B_logistica", "milho ativa + Arco Norte 2014 e Ferrovia 2013", "ativa")
L2_1p_s  <- rec(feols(y_agr ~ lrs_1p + xrs_1p + dist_min_t_100km + choc_ferrov_13 + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_1p_choq_soja", "caminho_B_logistica", "soja obs ln(1+rec) + choques logisticos", "obs")
L2_1p_m  <- rec(feols(y_agr ~ lrm_1p + xrm_1p + dist_min_t_100km + choc_ferrov_13 + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_1p_choq_milho", "caminho_B_logistica", "milho obs ln(1+rec) + choques logisticos", "obs")
cat("=== CAMINHO B1: Logística, Arco Norte (2014) e Ferrovia de Rondonópolis (2013) ===\n")
tab(L1_act_s, L2_act_s, L1_act_m, L2_act_m, L2_1p_s, L2_1p_m,
    headers = c("Soja ativa×Sul", "Soja ativa+Log", "Milho ativa×Sul", "Milho ativa+Log", "Soja obs+Log", "Milho obs+Log"))


=== CAMINHO B1: Logística, Arco Norte (2014) e Ferrovia de Rondonópolis (2013) ===


                      Soja ativa×Sul      Soja ativa+Log     Milho ativa×Sul     Milho ativa+Log        Soja obs+Log        Milho obs+Log
lrs_act              0.163* (0.0926)     0.0671 (0.0775)                                                                                 
xrs_act              0.0323 (0.0335)    0.0595* (0.0336)                                                                                 
xrs_act_proxsul     -0.0386 (0.0616)                                                                                                     
x_rs_act_proxsul     0.0492 (0.0502)                                                                                                     
zs_tr             0.0615*** (0.0213)  0.0807*** (0.0210)                                          0.0793*** (0.0225)                     
rain             -0.0157** (0.00633) -0.0161** (0.00644) -0.0151** (0.00668) -0.0171** (0.00686) -0.0137** (0.00581)  -0.0170** (0.00681)
dist_min_t_100km                  

***
##### 2.7 CAMINHO B2 — Embargos do IBAMA Geocodificados (`termo_embargo_historico.csv`) e Autos de Desmatamento (Etapa 14D)

Testamos a inclusão direta e moderadora dos embargos geocodificados do IBAMA em $t-1$ (número de embargos `emb_esp_l1` e área embargada em hectares `emb_area_l1`) e dos autos de desmatamento (`f_desm_l1`) nos modelos de rendimentos reais por hectare.


In [9]:
emb_path <- file.path(root, "xdata/chap1/raw/tabular/ibama/termo_embargo_historico.csv")
dt_emb <- fread(
  emb_path, sep = ";",
  select = c("SEQ_TAD", "SIT_CANCELADO", "DAT_EMBARGO", "COD_MUNICIPIO", "UF",
             "NUM_LONGITUDE_TAD", "NUM_LATITUDE_TAD", "QTD_AREA_EMBARGADA"),
  encoding = "UTF-8", showProgress = FALSE
)

dt_mt <- dt_emb[(UF == "MT" | substr(as.character(COD_MUNICIPIO), 1, 2) == "51") & SIT_CANCELADO != "S"]
dt_mt[, dt := as.Date(substr(DAT_EMBARGO, 1, 10))]
dt_mt[, yr := as.integer(format(dt, "%Y"))]
dt_mt <- dt_mt[!is.na(yr) & yr >= 2000 & yr <= 2024]
setorder(dt_mt, SEQ_TAD, dt)
dedup_emb <- unique(dt_mt, by = "SEQ_TAD")
dedup_emb[, area_ha := suppressWarnings(as.numeric(gsub(",", ".", QTD_AREA_EMBARGADA)))]
dedup_emb[is.na(area_ha) | area_ha < 0, area_ha := 0]

dedup_emb[, lon := suppressWarnings(as.numeric(gsub(",", ".", NUM_LONGITUDE_TAD)))]
dedup_emb[, lat := suppressWarnings(as.numeric(gsub(",", ".", NUM_LATITUDE_TAD)))]
pts_valid <- dedup_emb[!is.na(lon) & !is.na(lat) & lon >= -65 & lon <= -48 & lat >= -20 & lat <= -6]
sf_emb <- st_as_sf(pts_valid, coords = c("lon", "lat"), crs = st_crs(shp_mt))
joined_emb <- st_join(sf_emb, shp_mt["CD_MUN"], join = st_within)
st_geometry(joined_emb) <- NULL

grid_emb <- CJ(ibge_int = dist_mun$ibge_int, yr = 2000:2024)
emb_esp <- as.data.table(joined_emb)[!is.na(CD_MUN), .(
  n_emb_esp = .N,
  area_emb_esp_ha = sum(area_ha)
), by = .(ibge_int = as.integer(CD_MUN), yr)]

emb_panel <- merge(grid_emb, emb_esp, by = c("ibge_int", "yr"), all.x = TRUE)
emb_panel[is.na(n_emb_esp), `:=`(n_emb_esp = 0L, area_emb_esp_ha = 0)]
setorder(emb_panel, ibge_int, yr)
emb_panel[, `:=`(
  emb_esp_l1  = log1p(shift(n_emb_esp, 1L)),
  emb_area_l1 = log1p(shift(area_emb_esp_ha, 1L))
), by = ibge_int]

x3b_path <- file.path(root, "xdata/chap1/processed/harmonizacao/x3b_infracoes/instituicoes_espacial.csv")
x3b_desm <- fread(x3b_path, showProgress = FALSE) %>%
  as_tibble() %>%
  transmute(ibge_int = as.integer(codigo_ibge), ano = as.integer(ano), f_desm_l1 = log1p(n_autos_desmatamento_lag1))

d_camB <- d_camB %>%
  left_join(as_tibble(emb_panel) %>% rename(ano = yr), by = c("ibge_int", "ano")) %>%
  left_join(x3b_desm, by = c("ibge_int", "ano")) %>%
  mutate(
    emb_rs_act = lrs_act * emb_esp_l1, emb_zs = zs * emb_esp_l1, emb_xrs_act = xrs_act * emb_esp_l1,
    emb_area_xrs_act = xrs_act * emb_area_l1,
    emb_area_xrm_act = xrm_act * emb_area_l1,
    emb_area_xrs_1p  = xrs_1p  * emb_area_l1,
    emb_area_xrm_1p  = xrm_1p  * emb_area_l1
  )
d_camB_act_s <- d_camB %>% filter(act_s)
d_camB_act_m <- d_camB %>% filter(act_m)

B2_act_s1 <- rec(feols(y_agr ~ lrs_act + xrs_act + emb_esp_l1 + emb_rs_act + emb_zs + emb_xrs_act + zs_tr + rain | ibge + bioma^ano, d_camB_act_s, vcov = ~ibge),
                 "B2_act_emb_mod_soja", "caminho_B_embargos", "soja ativa + moderacao N embargos", "ativa")
B2_act_s2 <- rec(feols(y_agr ~ lrs_act + xrs_act + emb_area_l1 + emb_area_xrs_act + zs_tr + rain | ibge + bioma^ano, d_camB_act_s, vcov = ~ibge),
                 "B2_act_emb_area_soja", "caminho_B_embargos", "soja ativa + area embargada (ha) lag1", "ativa")
B2_act_m2 <- rec(feols(y_agr ~ lrm_act + xrm_act + emb_area_l1 + emb_area_xrm_act + zm_tr + rain | ibge + bioma^ano, d_camB_act_m, vcov = ~ibge),
                 "B2_act_emb_area_milho", "caminho_B_embargos", "milho ativa + area embargada (ha) lag1", "ativa")
B2_1p_s2  <- rec(feols(y_agr ~ lrs_1p + xrs_1p + emb_area_l1 + emb_area_xrs_1p + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
                 "B2_1p_emb_area_soja", "caminho_B_embargos", "soja obs ln(1+rec) + area embargada lag1", "obs")
B2_1p_m2  <- rec(feols(y_agr ~ lrm_1p + xrm_1p + emb_area_l1 + emb_area_xrm_1p + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
                 "B2_1p_emb_area_milho", "caminho_B_embargos", "milho obs ln(1+rec) + area embargada lag1", "obs")
B2_desm   <- rec(feols(y_agr ~ lrs_act + xrs_act + f_desm_l1 + zs_tr + rain | ibge + bioma^ano, d_camB_act_s, vcov = ~ibge),
                 "B2_act_autos_desm_soja", "caminho_B_embargos", "soja ativa + autos desmatamento lag1", "ativa c/ desm")

cat("=== CAMINHO B2: Embargos do IBAMA Geocodificados e Autos de Desmatamento ===\n")
tab(B2_act_s1, B2_act_s2, B2_act_m2, B2_1p_s2, B2_1p_m2, B2_desm,
    headers = c("Soja ativa×N Emb", "Soja ativa+Área", "Milho ativa+Área", "Soja obs+Área", "Milho obs+Área", "Soja+Autos Desm"))


=== CAMINHO B2: Embargos do IBAMA Geocodificados e Autos de Desmatamento ===


                    Soja ativa×N Emb       Soja ativa+Área      Milho ativa+Área         Soja obs+Área        Milho obs+Área    Soja+Autos Desm
lrs_act              0.181* (0.0919)       0.159* (0.0891)                                                                        0.216 (0.157)
xrs_act              0.0250 (0.0336)       0.0263 (0.0308)                                                                    0.119*** (0.0429)
emb_esp_l1           0.0191 (0.0144)                                                                                                           
emb_rs_act          -0.0168 (0.0254)                                                                                                           
emb_zs             0.0545** (0.0239)                                                                                                           
emb_xrs_act         -0.0138 (0.0372)                                                                                                    

***
##### 2.8 Cortes Temporais por Marcos Institucionais (`2001–2005`, `2006–2011`, `2012–2024`) $\times$ Biomas (Amazônia vs. Cerrado)

Decompomos o efeito da interação principal nos três regimes histórico-institucionais destacados no artigo, cruzando-os com os biomas predominantes (**Amazônia** vs. **Cerrado**), tanto para **H1-Preços** quanto para **H1-Rendimentos**:
- **Regime P1 (`2001–2005`):** Entrada da China na OMC (dez/2001) e boom de expansão agrícola pré-Moratória;
- **Regime P2 (`2006–2011`):** Início da Moratória da Soja na Amazônia (jul/2006) e consolidação do PPCDAm até a véspera do Novo Código Florestal;
- **Regime P3 (`2012–2024`):** Vigência do Novo Código Florestal (Lei nº 12.651, maio/2012) e expansão logística (Ferrovia Rondonópolis 2013 e Arco Norte 2014).


In [10]:
d_reg <- d_obs %>% mutate(
  d_p1 = as.numeric(ano <= 2005),
  d_p2 = as.numeric(ano >= 2006 & ano <= 2011),
  d_p3 = as.numeric(ano >= 2012),
  # Preços x Regime
  xs_p1 = xs * d_p1, xs_p2 = xs * d_p2, xs_p3 = xs * d_p3,
  xm_p1 = xm * d_p1, xm_p2 = xm * d_p2, xm_p3 = xm * d_p3,
  # Rendimentos Ativa (ln rec/ha) x Regime
  lrs_p1 = lrs_act * d_p1, lrs_p2 = lrs_act * d_p2, lrs_p3 = lrs_act * d_p3,
  xrs_p1 = xrs_act * d_p1, xrs_p2 = xrs_act * d_p2, xrs_p3 = xrs_act * d_p3,
  lrm_p1 = lrm_act * d_p1, lrm_p2 = lrm_act * d_p2, lrm_p3 = lrm_act * d_p3,
  xrm_p1 = xrm_act * d_p1, xrm_p2 = xrm_act * d_p2, xrm_p3 = xrm_act * d_p3,
  # Rendimentos Obs Completa ln(1+rec/ha) x Regime
  lrs1p_p1 = lrs_1p * d_p1, lrs1p_p2 = lrs_1p * d_p2, lrs1p_p3 = lrs_1p * d_p3,
  xrs1p_p1 = xrs_1p * d_p1, xrs1p_p2 = xrs_1p * d_p2, xrs1p_p3 = xrs_1p * d_p3,
  lrm1p_p1 = lrm_1p * d_p1, lrm1p_p2 = lrm_1p * d_p2, lrm1p_p3 = lrm_1p * d_p3,
  xrm1p_p1 = xrm_1p * d_p1, xrm1p_p2 = xrm_1p * d_p2, xrm1p_p3 = xrm_1p * d_p3
)

# 1. H1-Preços por Regime e Bioma
RP_s_mt  <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge), 'REG_prc_soja_MT', 'regimes_biomas', 'Preços Soja: 3 regimes MT', 'obs')
RP_s_amz <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge), 'REG_prc_soja_Amz', 'regimes_biomas', 'Preços Soja: 3 regimes Amazônia', 'obs Amazônia')
RP_s_cer <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge), 'REG_prc_soja_Cer', 'regimes_biomas', 'Preços Soja: 3 regimes Cerrado', 'obs Cerrado')
RP_m_mt  <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge), 'REG_prc_milho_MT', 'regimes_biomas', 'Preços Milho: 3 regimes MT', 'obs')
RP_m_amz <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge), 'REG_prc_milho_Amz', 'regimes_biomas', 'Preços Milho: 3 regimes Amazônia', 'obs Amazônia')
RP_m_cer <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge), 'REG_prc_milho_Cer', 'regimes_biomas', 'Preços Milho: 3 regimes Cerrado', 'obs Cerrado')
cat('=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-PREÇOS ===\n')
tab(RP_s_mt, RP_s_amz, RP_s_cer, RP_m_mt, RP_m_amz, RP_m_cer,
    headers = c('Soja MT', 'Soja Amazônia', 'Soja Cerrado', 'Milho MT', 'Milho Amazônia', 'Milho Cerrado'))

# 2. H1-Rendimentos (Lavoura Ativa) por Regime e Bioma
RR_s_mt  <- rec(feols(y_agr ~ lrs_p1 + lrs_p2 + lrs_p3 + xrs_p1 + xrs_p2 + xrs_p3 + rain | ibge + bioma^ano, filter(d_reg, act_s), vcov = ~ibge), 'REG_act_soja_MT', 'regimes_biomas', 'Rendimentos Ativa Soja: 3 regimes MT', 'ativa')
RR_s_amz <- rec(feols(y_agr ~ lrs_p1 + lrs_p2 + lrs_p3 + xrs_p1 + xrs_p2 + xrs_p3 + rain | ibge + ano, filter(d_reg, act_s & bioma == bioma_amz), vcov = ~ibge), 'REG_act_soja_Amz', 'regimes_biomas', 'Rendimentos Ativa Soja: 3 regimes Amazônia', 'ativa Amazônia')
RR_s_cer <- rec(feols(y_agr ~ lrs_p1 + lrs_p2 + lrs_p3 + xrs_p1 + xrs_p2 + xrs_p3 + rain | ibge + ano, filter(d_reg, act_s & bioma == bioma_cer), vcov = ~ibge), 'REG_act_soja_Cer', 'regimes_biomas', 'Rendimentos Ativa Soja: 3 regimes Cerrado', 'ativa Cerrado')
RR_m_mt  <- rec(feols(y_agr ~ lrm_p1 + lrm_p2 + lrm_p3 + xrm_p1 + xrm_p2 + xrm_p3 + rain | ibge + bioma^ano, filter(d_reg, act_m), vcov = ~ibge), 'REG_act_milho_MT', 'regimes_biomas', 'Rendimentos Ativa Milho: 3 regimes MT', 'ativa')
RR_m_amz <- rec(feols(y_agr ~ lrm_p1 + lrm_p2 + lrm_p3 + xrm_p1 + xrm_p2 + xrm_p3 + rain | ibge + ano, filter(d_reg, act_m & bioma == bioma_amz), vcov = ~ibge), 'REG_act_milho_Amz', 'regimes_biomas', 'Rendimentos Ativa Milho: 3 regimes Amazônia', 'ativa Amazônia')
RR_m_cer <- rec(feols(y_agr ~ lrm_p1 + lrm_p2 + lrm_p3 + xrm_p1 + xrm_p2 + xrm_p3 + rain | ibge + ano, filter(d_reg, act_m & bioma == bioma_cer), vcov = ~ibge), 'REG_act_milho_Cer', 'regimes_biomas', 'Rendimentos Ativa Milho: 3 regimes Cerrado', 'ativa Cerrado')
cat('\n=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-RENDIMENTOS (LAVOURA ATIVA) ===\n')
tab(RR_s_mt, RR_s_amz, RR_s_cer, RR_m_mt, RR_m_amz, RR_m_cer,
    headers = c('Soja MT', 'Soja Amazônia', 'Soja Cerrado', 'Milho MT', 'Milho Amazônia', 'Milho Cerrado'))

# 3. H1-Rendimentos (Amostra Completa ln(1+rec)) por Regime e Bioma
R1P_s_mt  <- rec(feols(y_agr ~ lrs1p_p1 + lrs1p_p2 + lrs1p_p3 + xrs1p_p1 + xrs1p_p2 + xrs1p_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge), 'REG_1p_soja_MT', 'regimes_biomas', 'Rendimentos Obs ln(1+rec) Soja: 3 regimes MT', 'obs')
R1P_s_amz <- rec(feols(y_agr ~ lrs1p_p1 + lrs1p_p2 + lrs1p_p3 + xrs1p_p1 + xrs1p_p2 + xrs1p_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge), 'REG_1p_soja_Amz', 'regimes_biomas', 'Rendimentos Obs ln(1+rec) Soja: 3 regimes Amazônia', 'obs Amazônia')
R1P_s_cer <- rec(feols(y_agr ~ lrs1p_p1 + lrs1p_p2 + lrs1p_p3 + xrs1p_p1 + xrs1p_p2 + xrs1p_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge), 'REG_1p_soja_Cer', 'regimes_biomas', 'Rendimentos Obs ln(1+rec) Soja: 3 regimes Cerrado', 'obs Cerrado')
R1P_m_mt  <- rec(feols(y_agr ~ lrm1p_p1 + lrm1p_p2 + lrm1p_p3 + xrm1p_p1 + xrm1p_p2 + xrm1p_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge), 'REG_1p_milho_MT', 'regimes_biomas', 'Rendimentos Obs ln(1+rec) Milho: 3 regimes MT', 'obs')
R1P_m_amz <- rec(feols(y_agr ~ lrm1p_p1 + lrm1p_p2 + lrm1p_p3 + xrm1p_p1 + xrm1p_p2 + xrm1p_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge), 'REG_1p_milho_Amz', 'regimes_biomas', 'Rendimentos Obs ln(1+rec) Milho: 3 regimes Amazônia', 'obs Amazônia')
R1P_m_cer <- rec(feols(y_agr ~ lrm1p_p1 + lrm1p_p2 + lrm1p_p3 + xrm1p_p1 + xrm1p_p2 + xrm1p_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge), 'REG_1p_milho_Cer', 'regimes_biomas', 'Rendimentos Obs ln(1+rec) Milho: 3 regimes Cerrado', 'obs Cerrado')
cat('\n=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-RENDIMENTOS (OBS ln(1+rec)) ===\n')
tab(R1P_s_mt, R1P_s_amz, R1P_s_cer, R1P_m_mt, R1P_m_amz, R1P_m_cer,
    headers = c('Soja MT', 'Soja Amazônia', 'Soja Cerrado', 'Milho MT', 'Milho Amazônia', 'Milho Cerrado'))


=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-PREÇOS ===


                          Soja MT      Soja Amazônia       Soja Cerrado            Milho MT     Milho Amazônia      Milho Cerrado
xs_p1            0.354*** (0.135)      0.400 (0.246)    0.339** (0.163)                                                          
xs_p2             0.0108 (0.0211) -0.0643** (0.0318)  0.0523** (0.0250)                                                          
xs_p3             0.0120 (0.0111)   0.00965 (0.0153)   0.00566 (0.0149)                                                          
rain          -0.0138** (0.00618) -0.0231** (0.0104) -0.00386 (0.00574) -0.0161** (0.00669) -0.0239** (0.0107) -0.00757 (0.00727)
xm_p1                                                                      0.423*** (0.121)      0.367 (0.329)   0.450*** (0.134)
xm_p2                                                                       0.0198 (0.0266)    -0.114 (0.0729)   0.0553* (0.0298)
xm_p3                                                                    -0.00407 (0.00767


=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-RENDIMENTOS (LAVOURA ATIVA) ===


                          Soja MT      Soja Amazônia       Soja Cerrado            Milho MT     Milho Amazônia      Milho Cerrado
lrs_p1           0.565*** (0.214)    1.23*** (0.426)    0.449** (0.215)                                                          
lrs_p2           -0.0580 (0.0990)     -0.164 (0.232)   -0.0208 (0.0766)                                                          
lrs_p3           -0.0287 (0.0812)     0.0387 (0.170)   -0.132* (0.0781)                                                          
xrs_p1           0.319*** (0.118)     1.45** (0.676)    0.236** (0.111)                                                          
xrs_p2            0.0470 (0.0294)    0.00235 (0.127)   0.0467* (0.0252)                                                          
xrs_p3            0.0195 (0.0261)     -0.155 (0.126)   0.0450* (0.0267)                                                          
rain          -0.0166** (0.00637) -0.0245** (0.0100) -0.00679 (0.00694) -0.0155** (0.00665


=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-RENDIMENTOS (OBS ln(1+rec)) ===


                           Soja MT       Soja Amazônia       Soja Cerrado            Milho MT     Milho Amazônia       Milho Cerrado
lrs1p_p1        0.0465*** (0.0142)   0.0593** (0.0252) 0.0400*** (0.0121)                                                           
lrs1p_p2      -0.00659** (0.00277) -0.00937* (0.00482) -0.00341 (0.00333)                                                           
lrs1p_p3        -0.00168 (0.00216)  1.58e-05 (0.00303)  -0.0120 (0.00804)                                                           
xrs1p_p1         -0.0110 (0.00692) -0.000633 (0.00371) -0.0140* (0.00831)                                                           
xrs1p_p2      0.00635*** (0.00198) 0.00655** (0.00296)  0.00469 (0.00366)                                                           
xrs1p_p3      0.00568*** (0.00206) 0.00576** (0.00278)  0.0133* (0.00740)                                                           
rain           -0.0156** (0.00609) -0.0217** (0.00924) -0.00759 (0.00

***
##### 2.9 Consolidação Geral, Veredito Pré-Especificado e Comparação Direta H1-Preços (`v2`) vs. H1-Rendimentos (`v1`) (Etapa 15)

Exporta todos os modelos estimados para `2_INTI/audit/modelos_h1_rendimentos_20261110/resultados_modelos_rendimentos.csv`, avalia os 6 critérios de decisão (`veredito_criterio_rendimentos.csv`) tanto na **amostra com lavoura ativa (`spec = 'act'`)** quanto na **amostra observada completa (`spec = '1p'`)**, e apresenta um quadro comparativo lado a lado com os resultados de **H1-Preços (`modelos_fixos_h1_prec_v2.ipynb`)**.


In [11]:
res_all <- bind_rows(RES)
write.csv(res_all, file.path(out_dir, "resultados_modelos_rendimentos.csv"), row.names = FALSE, fileEncoding = "UTF-8")
cat("Total exportado para resultados_modelos_rendimentos.csv:", n_distinct(res_all$id), "modelos/testes;", nrow(res_all), "linhas.\n")

get1 <- function(id, termo) res_all %>% filter(id == !!id, termo == !!termo) %>% slice(1)
veredito_r <- function(crop, sp = "act") {
  x  <- if (sp == "act") (if (crop == "soja") "xrs_act" else "xrm_act") else (if (crop == "soja") "xrs_1p" else "xrm_1p")
  xf <- paste0(x, "_f")
  m_id <- paste0(ifelse(crop == "soja", "M1_", "M2_"), sp)
  pri <- get1(m_id, x)
  cl2 <- get1(paste0("R_cl2_", sp, "_", crop), x)
  tws <- get1(paste0("R_2step_", sp, "_", crop), "lr")
  lo  <- res_all %>% filter(grepl(paste0("^R_loyo_", sp, "_", crop, "_"), id), termo == x)
  pl  <- get1(paste0("R_plac_", sp, "_", crop), xf)
  w06 <- get1(paste0("R_w06_", sp, "_", crop), x)
  crit <- tibble(
    criterio = c("C0 principal: estimativa > 0 e p < 0,10",
                 "C1 cluster município+ano: mesmo sinal e p < 0,10",
                 "C2 regressão em 24 pontos: mesmo sinal e p < 0,10",
                 "C3 exclusão de um ano: mesmo sinal em 24 e p < 0,10 em >= 19",
                 "C4 placebo: rendimento futuro com p >= 0,10",
                 "C5 2006-2024: mesmo sinal"),
    atendido = c(pri$estimativa > 0 & pri$p < 0.10,
                 sign(cl2$estimativa) == sign(pri$estimativa) & cl2$p < 0.10,
                 sign(tws$estimativa) == sign(pri$estimativa) & tws$p < 0.10,
                 all(sign(lo$estimativa) == sign(pri$estimativa)) & sum(lo$p < 0.10) >= 19,
                 pl$p >= 0.10,
                 sign(w06$estimativa) == sign(pri$estimativa)),
    valor = c(sprintf("est %.4f p %.3f", pri$estimativa, pri$p),
              sprintf("est %.4f p %.3f", cl2$estimativa, cl2$p),
              sprintf("est %.4f p %.3f", tws$estimativa, tws$p),
              sprintf("sinais iguais %d/24; p<0,10 em %d", sum(sign(lo$estimativa) == sign(pri$estimativa)), sum(lo$p < 0.10)),
              sprintf("est %.4f p %.3f", pl$estimativa, pl$p),
              sprintf("est %.4f p %.3f", w06$estimativa, w06$p))
  )
  classe <- if (!crit$atendido[1]) "Não consistente" else if (all(crit$atendido)) "Consistente" else "Parcial"
  list(crit = crit, classe = classe)
}

v_list <- list()
for (sp in c("act", "1p")) {
  for (crop in c("soja", "milho")) {
    v <- veredito_r(crop, sp)
    cat(sprintf("\n=== VEREDITO PRÉ-ESPECIFICADO — %s (%s): %s ===\n", toupper(crop), toupper(sp), v$classe))
    print(as.data.frame(v$crit), row.names = FALSE)
    v_list[[length(v_list) + 1]] <- v$crit %>% mutate(cultura = crop, especificacao = sp, classe = v$classe)
  }
}
write.csv(bind_rows(v_list), file.path(out_dir, "veredito_criterio_rendimentos.csv"), row.names = FALSE, fileEncoding = "UTF-8")

# Quadro comparativo direto: H1-Preços (v2) vs. H1-Rendimentos (v1)
prec_csv <- file.path(root, "audit", "modelos_h1_prec_v2_20261110", "resultados_modelos.csv")
if (file.exists(prec_csv)) {
  res_p <- as_tibble(read.csv(prec_csv, stringsAsFactors = FALSE))
  fmt_row <- function(df_src, id_val, t_val, lbl) {
    r <- df_src %>% filter(id == !!id_val, termo == !!t_val) %>% slice(1)
    if (nrow(r) == 0) return(NULL)
    sig <- ifelse(r$p < 0.01, "***", ifelse(r$p < 0.05, "**", ifelse(r$p < 0.10, "*", "")))
    tibble(Especificacao = lbl, Termo = t_val, Estimativa_EP = sprintf("%.4f%s (%.4f)", r$estimativa, sig, r$erro_padrao),
           p_valor = sprintf("%.4f", r$p), N = r$n)
  }
  comp <- bind_rows(
    fmt_row(res_p,   "M1",          "xs",           "1. SOJA — H1-Preços (M1: ln P x Z)"),
    fmt_row(res_all, "M1_act",      "xrs_act",      "2. SOJA — H1-Rendimentos Ativa c/ ef. princ. (ln Rec/ha x Z)"),
    fmt_row(res_all, "M1_act_nolv", "xrs_act",      "3. SOJA — H1-Rendimentos Ativa s/ ef. princ. (ln Rec/ha x Z)"),
    fmt_row(res_all, "D_both_act_soja", "xs",       "4a. SOJA — Decomposição Ativa: componente Preço (ln P x Z)"),
    fmt_row(res_all, "D_both_act_soja", "xyld_s_act","4b. SOJA — Decomposição Ativa: componente Produtividade PAM (ln Y x Z)"),
    fmt_row(res_all, "M1_1p",       "xrs_1p",       "5. SOJA — H1-Rendimentos Obs Completa (ln(1+Rec/ha) x Z)"),
    fmt_row(res_all, "M1_lvl",      "xrs_k",        "6. SOJA — H1-Rendimentos Obs Nível (USD 1.000/ha x Z)"),
    fmt_row(res_all, "L_act_soja",  "xluc_s_act_k", "7. SOJA — Lucro Operacional Líquido Real/ha Ativa (USD 1.000/ha x Z)"),
    fmt_row(res_all, "L_full_soja", "xluc_s_full_k","8. SOJA — Lucro Operacional Líquido Real/ha Obs (USD 1.000/ha x Z)"),
    fmt_row(res_p,   "M2",          "xm",           "9. MILHO — H1-Preços (M2: ln P x Z)"),
    fmt_row(res_all, "M2_act",      "xrm_act",      "10. MILHO — H1-Rendimentos Ativa c/ ef. princ. (ln Rec/ha x Z)"),
    fmt_row(res_all, "M2_act_nolv", "xrm_act",      "11. MILHO — H1-Rendimentos Ativa s/ ef. princ. (ln Rec/ha x Z)"),
    fmt_row(res_all, "D_both_act_milho", "xm",      "12a. MILHO — Decomposição Ativa: componente Preço (ln P x Z)"),
    fmt_row(res_all, "D_both_act_milho", "xyld_m_act","12b. MILHO — Decomposição Ativa: componente Produtividade PAM (ln Y x Z)"),
    fmt_row(res_all, "M2_1p",       "xrm_1p",       "13. MILHO — H1-Rendimentos Obs Completa (ln(1+Rec/ha) x Z)"),
    fmt_row(res_all, "M2_lvl",      "xrm_k",        "14. MILHO — H1-Rendimentos Obs Nível (USD 1.000/ha x Z)"),
    fmt_row(res_all, "L_act_milho", "xluc_m_act_k", "15. MILHO — Lucro Operacional Líquido Real/ha Ativa (USD 1.000/ha x Z)"),
    fmt_row(res_all, "L_full_milho","xluc_m_full_k","16. MILHO — Lucro Operacional Líquido Real/ha Obs (USD 1.000/ha x Z)")
  )
  cat("\n=== SÍNTESE COMPARATIVA: H1-PREÇOS (v2) vs. H1-RENDIMENTOS (v1) ===\n")
  print(as.data.frame(comp), row.names = FALSE)
}


Total exportado para resultados_modelos_rendimentos.csv: 258 modelos/testes; 1113 linhas.



=== VEREDITO PRÉ-ESPECIFICADO — SOJA (ACT): Não consistente ===
                                                     criterio atendido                            valor
                      C0 principal: estimativa > 0 e p < 0,10    FALSE               est 0.0392 p 0.210
             C1 cluster município+ano: mesmo sinal e p < 0,10    FALSE               est 0.0392 p 0.466
            C2 regressão em 24 pontos: mesmo sinal e p < 0,10     TRUE               est 0.1918 p 0.057
 C3 exclusão de um ano: mesmo sinal em 24 e p < 0,10 em >= 19    FALSE sinais iguais 24/24; p<0,10 em 2
                  C4 placebo: rendimento futuro com p >= 0,10    FALSE               est 0.1199 p 0.014
                                    C5 2006-2024: mesmo sinal    FALSE              est -0.0118 p 0.265

=== VEREDITO PRÉ-ESPECIFICADO — MILHO (ACT): Parcial ===
                                                     criterio atendido                             valor
                      C0 principal: estimati


=== SÍNTESE COMPARATIVA: H1-PREÇOS (v2) vs. H1-RENDIMENTOS (v1) ===
                                                            Especificacao         Termo       Estimativa_EP p_valor    N
                                       1. SOJA — H1-Preços (M1: ln P x Z)            xs   0.0848** (0.0358)  0.0192 2767
             2. SOJA — H1-Rendimentos Ativa c/ ef. princ. (ln Rec/ha x Z)       xrs_act     0.0392 (0.0311)  0.2097 2091
             3. SOJA — H1-Rendimentos Ativa s/ ef. princ. (ln Rec/ha x Z)       xrs_act    0.0544* (0.0277)  0.0517 2091
               4a. SOJA — Decomposição Ativa: componente Preço (ln P x Z)            xs   0.1322** (0.0506)  0.0100 2091
   4b. SOJA — Decomposição Ativa: componente Produtividade PAM (ln Y x Z)    xyld_s_act -0.2155*** (0.0757)  0.0052 2091
                 5. SOJA — H1-Rendimentos Obs Completa (ln(1+Rec/ha) x Z)        xrs_1p    -0.0034 (0.0031)  0.2725 2767
                    6. SOJA — H1-Rendimentos Obs Nível (USD 1.000/ha x Z)         xr